# Diverse ENSEMBLE 

Our final solution is a diverse ensemble. We combined per-rule specialists (DeBERTa NLI training; embedding models with logistic regression + k-NN), a train-only large model (Qwen-32B trained on the official data and inferred with similarity-based sampling), and several test-time-trained (TTT) components across all rules (public DeBERTa training; public FAISS + triplet pipeline; Qwen-3 8B full finetune; Llama-3.1 8B LoRA). All models used a consistent instruction-only prompt and Yes/No-constrained decoding, then we blended them with per-rule rank-normalized weighting.


# Installation 

In [1]:
!uv pip install --system --no-index --find-links='/kaggle/input/lib-download-kaggle/whls/' 'trl==0.19.0' 'bitsandbytes==0.46.1' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0' 'deepspeed==0.17.4'
!uv pip install --system --no-index --find-links='/kaggle/input/lib-download-kaggle/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/lib-download-kaggle/whls/' 'clean-text' 'faiss-cpu'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/lib-download-kaggle/whls/' 'peft' 'accelerate' 'datasets'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/lib-download-kaggle/whls/' 'numpy==1.26.4'

Using Python 3.11.13 environment at: /usr
Resolved 163 packages in 1.33s
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17.4
   Building deepspeed==0.17

# Embed models 

Multi-model ensemble with rule-specific tuning for binary classification of rule violations. Each rule gets its own customized classifier.

### Embedding Models (6 total)

- E5 (base, large), Qwen 3 (0.6B), BGE (small, large), GTE (large)
- All normalized to L2 norm for cosine similarity

### Per-Rule Pipeline

1. Duplicate : Groups similar examples (cosine sim > 0.95-0.97) and downweights them to prevent overfitting
2. Hyperparameter Tuning (6-fold CV):
   - Logistic Regression: 31 C-values tested
   - KNN: 10-15 K-values tested
   - Blends both (70% LR + 30% KNN)
3. Ensemble Weight Search: GPU-accelerated simplex grid search across all 6 models to find optimal weights for each rule
Stabilization: 90% ensemble score + 10% rank-mean blend to smooth predictions

In [2]:
%%writefile embed_model_lr.py


import os, gc, random, math, numpy as np, pandas as pd, torch, warnings
from typing import List, Dict, Tuple
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sentence_transformers import SentenceTransformer
from joblib import Parallel, delayed

warnings.filterwarnings('ignore')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ================ PATHS / IO =================
TEST_PATH = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
SUBMISSION_PATH = "submission_embed_model_0_909.csv"

GPU_COUNT = torch.cuda.device_count()

# ================ MODELS (same set as 0.909) =================
EMBEDDING_MODELS = {
    'e5_base':     {'path': "/kaggle/input/download-models-jigsaw/intfloat_base", 'device': 'cuda:1', 'batch_size': 64},
    'e5_large':    {'path': "/kaggle/input/download-models-jigsaw/intfloat_large", 'device': 'cuda:0', 'batch_size': 48},
    'qwen3_0_6b':  {'path': "/kaggle/input/qwen-3-embedding/transformers/0.6b/1", 'device': 'cuda:0', 'batch_size': 32},
    # 'qwen3_4b':    {'path': "/kaggle/input/qwen-3-embedding/transformers/4b/1",   'device': 'cuda:1', 'batch_size': 8,  'torch_dtype': torch.float16},
    'bge_small':   {'path': "/kaggle/input/download-models-jigsaw/bge_small",      'device': 'cuda:0', 'batch_size': 256},
    'bge_large':   {'path': "/kaggle/input/download-models-jigsaw/bge_base",       'device': 'cuda:0', 'batch_size': 64},
    'gte_large':   {'path': "/kaggle/input/download-models-jigsaw/gte_large",      'device': 'cuda:1', 'batch_size': 64, 'torch_dtype': torch.float16},
}
for k, cfg in EMBEDDING_MODELS.items():
    if 'cuda' in cfg['device']:
        idx = int(cfg['device'].split(':')[1])
        if idx >= GPU_COUNT:
            cfg['device'] = 'cuda:0' if GPU_COUNT >= 1 else 'cpu'

# ================ HPARAMS (v17 spirit) =================
C_VALUES = [
    0.03, 0.05, 0.08, 0.12, 0.18, 0.25, 0.35, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0,
    3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 12.0, 14.0, 16.0, 18.0,
    20.0, 24.0, 28.0, 32.0, 40.0, 50.0
]
ENSEMBLE_WEIGHT_SEARCH = True
WEIGHT_GRID = [ 0.02, 0.05, 0.1, 0.12, 0.15, 0.16, 0.18, 0.2, 0.22, 0.24, 0.25, 0.28, 0.3, 0.32, 0.35, 0.38,
               0.4, 0.42, 0.45, 0.5, 0.55, 0.6, 0.62, 0.65, 0.7, 0.75, 0.8, 0.85, 0.87, 0.88, 0.9, 0.91, 0.92, 0.94, 
               0.95, 0.97, 0.98 , 0.99]

ALPHA_LR = 0.70
N_FOLDS_HP_TUNE = 6
K_OPTIONS = [1, 3, 5, 8, 10, 15, 20, 25, 30, 50]

# ================ KNN + DUPLICATE-AWARE WEIGHTS (your asks) ================
KNN_K = 10
KNN_METRIC = "cosine"

DUP_COS_THRESH_MAIN = 0.95
DUP_COS_THRESH_TINY = 0.97
WEIGHT_MIN = 0.25

# ================ UTILS =================
def empty_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try: torch.cuda.ipc_collect()
        except Exception: pass

def l2_norm(X, eps=1e-12):
    if X.ndim == 1:
        n = np.linalg.norm(X) + eps
        return X / n
    n = np.linalg.norm(X, axis=1, keepdims=True) + eps
    return X / n

def _prefix_for_model(key: str, texts: List[str]) -> List[str]:
    return [f"passage: {t}" for t in texts] if 'e5' in key else texts

def duplicate_weights_greedy(Emb: np.ndarray) -> np.ndarray:
    n = Emb.shape[0]
    if n <= 1:
        return np.ones(n, dtype=np.float32)
    thr = DUP_COS_THRESH_TINY if n < 20 else DUP_COS_THRESH_MAIN
    unassigned = np.ones(n, dtype=bool)
    cluster_id = np.full(n, -1, dtype=int)
    cid = 0
    while unassigned.any():
        i = np.argmax(unassigned)
        sims = Emb @ Emb[i]  # L2-normed -> dot == cosine
        in_cluster = (sims >= thr) & unassigned
        cluster_id[in_cluster] = cid
        unassigned[in_cluster] = False
        cid += 1
    sizes = np.bincount(cluster_id, minlength=cid).astype(np.float32)
    w = 1.0 / sizes[cluster_id]
    w = np.maximum(w, WEIGHT_MIN).astype(np.float32)
    w *= (n / (w.sum() + 1e-12))
    return w.astype(np.float32)

# ================ LOAD DATA =================
print("Loading data…")
df = pd.read_csv(TEST_PATH)
id_col = "row_id" if "row_id" in df.columns else ("id" if "id" in df.columns else None)
if id_col is None:
    df["row_id"] = range(len(df)); id_col = "row_id"

for c in ["rule","body","positive_example_1","positive_example_2","negative_example_1","negative_example_2"]:
    if c not in df.columns: df[c] = ""
df["rule"] = df["rule"].astype(str)

if len(df) == 10:
    N_FOLDS_HP_TUNE = 2
    K_OPTIONS =[1,2]
else:
    N_FOLDS_HP_TUNE = 6
    K_OPTIONS = [1, 2, 3, 5, 8, 10, 15, 20,  25, 30, 35, 40, 50, 60, 80]



rules = sorted(df["rule"].unique().tolist())
rule_to_index = {r:i for i,r in enumerate(rules)}
rule_to_row_indices = {r: np.where(df["rule"].values == r)[0] for r in rules}

ex_pos_cols = [c for c in df.columns if "positive_example" in c]
ex_neg_cols = [c for c in df.columns if "negative_example" in c]

rule_to_pos_texts, rule_to_neg_texts, rule_to_labels = {}, {}, {}
for r in rules:
    rdf = df[df["rule"] == r]
    pos, neg = [], []
    for c in ex_pos_cols: pos += rdf[c].dropna().astype(str).tolist()
    for c in ex_neg_cols: neg += rdf[c].dropna().astype(str).tolist()
    pos = list(dict.fromkeys([s.strip() for s in pos if s and s.strip()]))
    neg = list(dict.fromkeys([s.strip() for s in neg if s and s.strip()]))
    s_pos, s_neg = set(pos), set(neg)
    pos = [x for x in pos if x not in s_neg]
    neg = [x for x in neg if x not in s_pos]
    rule_to_pos_texts[r] = pos
    rule_to_neg_texts[r] = neg
    rule_to_labels[r] = np.array([1]*len(pos) + [0]*len(neg), dtype=np.int8)

# ================ ENCODING CACHE =================
MODEL_CACHE: Dict[str, Dict[str, np.ndarray]] = {}

def _load_model(cfg):
    kwargs = {}
    if 'torch_dtype' in cfg:
        kwargs['model_kwargs'] = {'torch_dtype': cfg['torch_dtype']}
    return SentenceTransformer(cfg['path'], device=cfg['device'], **kwargs)

def _encode_model_all(model_key: str, cfg: Dict):
    print(f"\nEncoding all for model: {model_key} on {cfg['device']}")
    embedder = _load_model(cfg)
    batch = cfg['batch_size']

    bodies_text = df["body"].astype(str).fillna("").tolist()
    bodies_enc = embedder.encode(_prefix_for_model(model_key, bodies_text),
                                 batch_size=batch, convert_to_numpy=True,
                                 show_progress_bar=True, normalize_embeddings=True).astype(np.float32)
    bodies_enc = l2_norm(bodies_enc)
    dim = bodies_enc.shape[1]
    store = {'dim': dim, 'body': bodies_enc}

    for r in rules:
        pe = rule_to_pos_texts[r]; ne = rule_to_neg_texts[r]
        if len(pe):
            P = embedder.encode(_prefix_for_model(model_key, pe),
                                batch_size=batch, convert_to_numpy=True,
                                show_progress_bar=False, normalize_embeddings=True).astype(np.float32)
            P = l2_norm(P)
        else:
            P = np.zeros((0, dim), dtype=np.float32)
        if len(ne):
            N = embedder.encode(_prefix_for_model(model_key, ne),
                                batch_size=batch, convert_to_numpy=True,
                                show_progress_bar=False, normalize_embeddings=True).astype(np.float32)
            N = l2_norm(N)
        else:
            N = np.zeros((0, dim), dtype=np.float32)
        ridx = rule_to_index[r]
        store[f'pos_{ridx}'] = P
        store[f'neg_{ridx}'] = N

    MODEL_CACHE[model_key] = store
    del embedder
    empty_cache()

for mkey, cfg in EMBEDDING_MODELS.items():
    _encode_model_all(mkey, cfg)

def get_or_encode_examples(rule: str, _unused_texts: List[str], model_key: str) -> np.ndarray:
    ridx = rule_to_index[rule]
    P = MODEL_CACHE[model_key].get(f'pos_{ridx}', None)
    N = MODEL_CACHE[model_key].get(f'neg_{ridx}', None)
    if P is None or N is None:
        dim = MODEL_CACHE[model_key]['dim']
        return np.zeros((0, dim), dtype=np.float32)
    if P.size == 0 and N.size == 0:
        dim = MODEL_CACHE[model_key]['dim']
        return np.zeros((0, dim), dtype=np.float32)
    return np.vstack([P, N]).astype(np.float32)

def get_body_block(row_indices: np.ndarray, model_key: str) -> np.ndarray:
    return MODEL_CACHE[model_key]['body'][row_indices]

# ================ FAST KNN HELPERS =================
def _knn_probs_from_neighbors(dists, nn_idx, y_tr, K_values, eps=1e-6):
    labels = y_tr[nn_idx]
    w = 1.0 / (dists + eps)
    num_cum = (w * labels).cumsum(axis=1)
    den_cum = w.cumsum(axis=1)
    out = {}
    k_max = dists.shape[1]
    for K in K_values:
        Kc = min(K, k_max)
        out[K] = (num_cum[:, Kc-1] / (den_cum[:, Kc-1] + 1e-12)).astype(np.float32)
    return out

# ================ TUNING (with duplicate-aware weights for LR) =================
def tune_lr_knn_for_rule(rule: str, labels: List[int]) -> Dict[str, Dict[str, float]]:
    print(f"  Tuning LR+KNN for rule '{rule[:60]}'...")
    best = {}
    if len(set(labels)) < 2:
        return {k: {"C": 1.0, "K": 5} for k in MODEL_CACHE.keys()}

    y = np.array(labels, dtype=np.int8)
    ridx = rule_to_index[rule]
    skf = StratifiedKFold(n_splits=N_FOLDS_HP_TUNE, shuffle=True, random_state=SEED)
    folds = list(skf.split(np.zeros(len(y)), y))

    for model_key in MODEL_CACHE.keys():
        P = MODEL_CACHE[model_key][f'pos_{ridx}']
        N = MODEL_CACHE[model_key][f'neg_{ridx}']
        if (len(P) + len(N)) == 0:
            best[model_key] = {"C": 1.0, "K": 5}
            continue
        X = np.vstack([P, N]).astype(np.float32)
        y_all = np.array([1]*len(P) + [0]*len(N), dtype=np.int8)

        # duplicate-aware weights for LR (train folds only)
        w_pos = duplicate_weights_greedy(P) if len(P) else np.ones(0, np.float32)
        w_neg = duplicate_weights_greedy(N) if len(N) else np.ones(0, np.float32)
        w_all = np.concatenate([w_pos, w_neg]).astype(np.float32)

        cv_scores = {(C,K): [] for C in C_VALUES for K in K_OPTIONS}
        k_max = max(K_OPTIONS)

        for tr_idx, va_idx in folds:
            X_tr, X_va = X[tr_idx], X[va_idx]
            y_tr, y_va = y_all[tr_idx], y_all[va_idx]
            w_tr = w_all[tr_idx]  # weights only on train portion

            # LR predictions for all Cs in parallel
            def _fit_pred_lr(C):
                lr = LogisticRegression(
                    C=C, penalty='l2', max_iter=1000, random_state=SEED,
                    class_weight="balanced", solver="lbfgs", n_jobs=1
                ).fit(X_tr, y_tr, sample_weight=w_tr)
                return C, lr.predict_proba(X_va)[:, 1].astype(np.float32)
            lr_results = Parallel(n_jobs=-1, prefer="threads")(
                delayed(_fit_pred_lr)(C) for C in C_VALUES
            )
            p_lr_dict = {C: p for C, p in lr_results}

            # KNN probs for all K in one neighbor query (cosine)
            nbrs = NearestNeighbors(n_neighbors=min(k_max, X_tr.shape[0]),
                                    metric=KNN_METRIC, algorithm='brute')
            nbrs.fit(X_tr)
            dists, nn_idx = nbrs.kneighbors(X_va, return_distance=True)
            p_knn_allK = _knn_probs_from_neighbors(dists, nn_idx, y_tr, K_OPTIONS)

            for C in C_VALUES:
                p_lr = p_lr_dict[C]
                for K in K_OPTIONS:
                    p_knn = p_knn_allK[K]
                    p_blend = ALPHA_LR * p_lr + (1.0 - ALPHA_LR) * p_knn
                    cv_scores[(C,K)].append(roc_auc_score(y_va, p_blend))

            del nbrs, dists, nn_idx, p_knn_allK, lr_results, p_lr_dict
            gc.collect()

        best_score, best_c, best_k = -1.0, C_VALUES[0], K_OPTIONS[0]
        for (C,K), arr in cv_scores.items():
            m = float(np.mean(arr)) if arr else 0.5
            if m > best_score:
                best_score, best_c, best_k = m, C, K
        best[model_key] = {"C": best_c, "K": best_k}
        print(f"      {model_key}: best C={best_c}, K={best_k}")
    return best

def _oof_by_model(rule: str, labels: List[int], hp: Dict[str, Dict[str, float]]):
    y = np.array(labels, dtype=np.int8)
    ridx = rule_to_index[rule]
    skf = StratifiedKFold(n_splits=N_FOLDS_HP_TUNE, shuffle=True, random_state=SEED)
    folds = list(skf.split(np.zeros(len(y)), y))

    X_by_model = {}
    y_by_model = {}
    w_by_model = {}
    for model_key in MODEL_CACHE.keys():
        P = MODEL_CACHE[model_key][f'pos_{ridx}']
        N = MODEL_CACHE[model_key][f'neg_{ridx}']
        X = np.vstack([P, N]).astype(np.float32)
        X_by_model[model_key] = X
        y_by_model[model_key] = np.array([1]*len(P) + [0]*len(N), dtype=np.int8)
        w_pos = duplicate_weights_greedy(P) if len(P) else np.ones(0, np.float32)
        w_neg = duplicate_weights_greedy(N) if len(N) else np.ones(0, np.float32)
        w_by_model[model_key] = np.concatenate([w_pos, w_neg]).astype(np.float32)

    oof_preds = {k: np.zeros(len(y_by_model[k]), dtype=np.float32) for k in MODEL_CACHE.keys()}
    aucs = {}

    for tr_idx, va_idx in folds:
        for model_key in MODEL_CACHE.keys():
            X = X_by_model[model_key]
            y_all = y_by_model[model_key]
            w_all = w_by_model[model_key]
            X_tr, X_va = X[tr_idx], X[va_idx]
            y_tr, y_va = y_all[tr_idx], y_all[va_idx]
            w_tr = w_all[tr_idx]

            C = hp[model_key]["C"]
            lr = LogisticRegression(
                C=C, penalty='l2', max_iter=1000, random_state=SEED,
                class_weight='balanced', solver="lbfgs", n_jobs=1
            ).fit(X_tr, y_tr, sample_weight=w_tr)
            p_lr = lr.predict_proba(X_va)[:, 1].astype(np.float32)

            K = int(hp[model_key]["K"])
            nbrs = NearestNeighbors(n_neighbors=min(K, X_tr.shape[0]),
                                    metric=KNN_METRIC, algorithm='brute')
            nbrs.fit(X_tr)
            dists, nn_idx = nbrs.kneighbors(X_va, return_distance=True)
            # distance-weighted labels (no sample weights in KNN)
            labels_tr = y_tr
            w = 1.0 / (dists + 1e-6)
            w = w / (w.sum(axis=1, keepdims=True) + 1e-12)
            p_knn = (w * labels_tr[nn_idx]).sum(axis=1).astype(np.float32)

            oof_preds[model_key][va_idx] = (ALPHA_LR * p_lr + (1.0 - ALPHA_LR) * p_knn).astype(np.float32)

            del lr, nbrs, dists, nn_idx, p_knn
        gc.collect()

    for k in MODEL_CACHE.keys():
        try:
            aucs[k] = roc_auc_score(y_by_model[k], oof_preds[k])
        except ValueError:
            aucs[k] = 0.5
    return oof_preds, aucs

# ================ GPU WEIGHT SEARCH (safe shortlist) =================
def _gen_simplex_grid(values, m, tol=1e-2):
    vals = sorted(values)
    acc = []
    def rec(idx, rem, chosen):
        if idx == m - 1:
            v = rem
            for cand in vals:
                if abs(cand - v) <= tol:
                    acc.append(tuple(chosen + [cand]))
                    break
            return
        for cand in vals:
            if cand > rem + tol: break
            rec(idx + 1, rem - cand, chosen + [cand])
    rec(0, 1.0, [])
    return acc

def _batch_scores_gpu(P_np: np.ndarray, weights: list[tuple], batch_size: int = 65536, device: str = "cuda"):
    P = torch.from_numpy(P_np).to(device, non_blocking=True)
    n, M = P.shape
    start = 0
    while start < len(weights):
        end = min(start + batch_size, len(weights))
        W = torch.tensor(weights[start:end], dtype=torch.float32, device=device).t()  # (M, B)
        S = P @ W  # (n, B)
        yield S, (start, end)
        start = end

@torch.inference_mode()
def _approx_auc_gpu(y_np: np.ndarray, S_gpu: torch.Tensor):
    device = S_gpu.device
    y = torch.from_numpy(y_np.astype(np.int64)).to(device, non_blocking=True)
    n, B = S_gpu.shape
    if n == 0 or B == 0:
        return torch.zeros((B,), device=device)
    idx = torch.argsort(S_gpu, dim=0)  # (n,B)
    ranks = torch.empty_like(S_gpu, dtype=torch.float32)
    arange = torch.arange(1, n+1, device=device, dtype=torch.float32).unsqueeze(1).expand(n, B)
    ranks.scatter_(0, idx, arange)
    y_f = y.float().unsqueeze(1)
    pos_rank_sum = (ranks * y_f).sum(dim=0)
    n_pos = int(y.sum().item())
    n_neg = n - n_pos
    if n_pos == 0 or n_neg == 0:
        return torch.full((B,), 0.5, device=device)
    auc = (pos_rank_sum - n_pos*(n_pos+1)/2.0) / (n_pos * n_neg)
    return auc  # (B,)

def _entropy(w):
    return -sum(float(x) * math.log(float(x) + 1e-12) for x in w)

def find_ensemble_weights_gpu(rule: str,
                              labels: List[int],
                              hp: Dict[str, Dict[str,float]],
                              *,
                              tol: float = 0.01,
                              topk_refine: int = 5000,
                              batch_size: int = 65536) -> Dict[str, float]:
    print(f"  Ensemble weight search for '{rule[:60]}' (GPU)…")
    oof_preds, _ = _oof_by_model(rule, labels, hp)
    keys = list(MODEL_CACHE.keys())
    y = np.asarray(labels, dtype=np.int8)
    P = np.stack([oof_preds[k] for k in keys], axis=1).astype(np.float32)
    M = P.shape[1]

    # candidate set: NO zeros, tol=0.01 (same surface as v17 good runs)
    cand_weights = _gen_simplex_grid(WEIGHT_GRID, M, tol=tol)
    n_cand = len(cand_weights)
    print(f"    candidates: {n_cand} (M={M}, tol={tol}, no zeros)")

    if n_cand == 0:
        return {k: 1.0/M for k in keys}

    device = "cuda" if torch.cuda.is_available() else "cpu"
    approx_scores = []
    with torch.inference_mode():
        for S, _slice in _batch_scores_gpu(P, cand_weights, batch_size=batch_size, device=device):
            auc_batch = _approx_auc_gpu(y, S)
            approx_scores.append(auc_batch.detach().cpu().numpy())
            del S, auc_batch
            if device.startswith("cuda"):
                torch.cuda.empty_cache()
    approx_scores = np.concatenate(approx_scores, axis=0)

    k = min(topk_refine, n_cand)
    top_idx = np.argpartition(-approx_scores, k-1)[:k]

    # Exact CPU refine + entropy tie-break among ties
    best_auc = -1.0; best_w = None; best_ent = -1e9
    P_cpu = P  # already on CPU
    for idx in top_idx:
        w = cand_weights[idx]
        s = P_cpu @ np.array(w, dtype=np.float32)
        auc = roc_auc_score(y, s)
        if auc > best_auc:
            best_auc, best_w, best_ent = auc, w, _entropy(w)
        elif abs(auc - best_auc) <= 1e-6:
            ent = _entropy(w)
            if ent > best_ent:
                best_w, best_ent = w, ent

    out = dict(zip(keys, map(float, best_w)))
    print(f"    best weights: {out}  (AUC={best_auc:.4f}) [GPU shortlist + CPU exact {k}, entropy-tie]")
    return out

# ================ ENSEMBLE =================
class RuleSpecificEnsemble:
    def __init__(self, rule: str):
        self.rule = rule
        self.hp = {}
        self.models = {}
        self.weights = {}

    def fit(self, texts_unused: List[str], labels: List[int]):
        print(f"Training ensemble for rule '{self.rule[:60]}' with {sum(labels)} pos / {len(labels)-sum(labels)} neg")
        self.hp = tune_lr_knn_for_rule(self.rule, labels)
        self.weights = find_ensemble_weights_gpu(self.rule, labels, self.hp, tol=0.01, topk_refine=10_000)

        # final per-model estimators
        for model_key in EMBEDDING_MODELS.keys():
            X = get_or_encode_examples(self.rule, texts_unused, model_key)
            if X.shape[0] == 0: 
                continue
            ridx = rule_to_index[self.rule]
            # final sample weights for full LR fit
            P = MODEL_CACHE[model_key][f'pos_{ridx}']
            N = MODEL_CACHE[model_key][f'neg_{ridx}']
            y_all = np.array([1]*len(P) + [0]*len(N), dtype=np.int8)
            w_pos = duplicate_weights_greedy(P) if len(P) else np.ones(0, np.float32)
            w_neg = duplicate_weights_greedy(N) if len(N) else np.ones(0, np.float32)
            w_all = np.concatenate([w_pos, w_neg]).astype(np.float32)

            lr = LogisticRegression(
                C=self.hp[model_key]["C"], penalty='l2', max_iter=1000, random_state=SEED,
                class_weight='balanced', solver="lbfgs", n_jobs=1
            ).fit(X, y_all, sample_weight=w_all)
            knn = KNeighborsClassifier(
                n_neighbors=int(self.hp[model_key]["K"]), metric=KNN_METRIC,
                algorithm='brute', weights='distance', n_jobs=1
            ).fit(X, y_all)
            self.models[model_key] = {"lr": lr, "knn": knn}
            print(f"  {model_key}: C={self.hp[model_key]['C']}, K={self.hp[model_key]['K']}, w={self.weights.get(model_key,0):.3f}")

    def predict_proba(self, row_indices: np.ndarray) -> np.ndarray:
        preds_by_model = {}
        for model_key, mk in self.models.items():
            Xb = get_body_block(row_indices, model_key)
            p_lr = mk["lr"].predict_proba(Xb)[:, 1]
            p_knn = mk["knn"].predict_proba(Xb)[:, 1]
            preds_by_model[model_key] = (ALPHA_LR * p_lr + (1.0 - ALPHA_LR) * p_knn).astype(np.float32)
            del Xb
        gc.collect()

        total_w = sum(self.weights.values()) or 1.0
        ens = np.zeros(len(row_indices), dtype=np.float32)
        for k, p in preds_by_model.items():
            ens += (self.weights.get(k, 0.0) / total_w) * p

        # Restore your v17 stabilizer: 10% rank mean
        if len(preds_by_model) >= 2:
            n = len(row_indices)
            rank_mean = np.zeros(n, dtype=np.float32)
            for p in preds_by_model.values():
                order = np.argsort(p)
                ranks = np.empty_like(order, dtype=np.float32)
                ranks[order] = np.arange(n, dtype=np.float32) / max(1, n-1)
                rank_mean += ranks
            rank_mean /= len(preds_by_model)
            ens = 0.90 * ens + 0.10 * rank_mean

        ens = np.clip(ens, 1e-6, 1-1e-6)
        return np.column_stack([1 - ens, ens])

# ================ SUBMISSION =================
def generate_submission():
    print("\n" + "="*70)
    print("GENERATING SUBMISSION (cached embeds, v20)")
    print("="*70)
    preds = np.zeros(len(df), dtype=np.float32)

    for i, rule in enumerate(rules):
        print(f"\n[{i+1}/{len(rules)}] Rule: {rule[:80]}")
        ex_texts = rule_to_pos_texts[rule] + rule_to_neg_texts[rule]
        ex_labels = rule_to_labels[rule].tolist()
        print(f"  Canonicals: {int(sum(ex_labels))} pos / {len(ex_labels)-int(sum(ex_labels))} neg")

        if len(set(ex_labels)) < 2:
            preds[rule_to_row_indices[rule]] = 0.5
            continue

        ens = RuleSpecificEnsemble(rule)
        ens.fit(ex_texts, ex_labels)

        row_idxs = rule_to_row_indices[rule]
        print(f"  Predicting {len(row_idxs)} samples...")
        p = ens.predict_proba(row_idxs)
        preds[row_idxs] = p[:, 1]
        del ens
        gc.collect()

    sub = pd.DataFrame({
        "row_id": df[id_col].values,
        "rule": df["rule"].astype(str).values,
        "rule_violation": np.clip(preds, 1e-6, 1 - 1e-6),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    print(f"\n✅ Saved {SUBMISSION_PATH} | mean={preds.mean():.4f} | rng=[{preds.min():.4f},{preds.max():.4f}]  shape={sub.shape}")
    return sub

if __name__ == "__main__":
    submission_df = generate_submission()

Writing embed_model_lr.py


# Faiss triplet 
- Include train and increase epoch

In [3]:
%%writefile triplet_faiss_public.py

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import random
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    models
)
from sentence_transformers.losses import TripletLoss
from sklearn.metrics.pairwise import cosine_similarity
import re
from urllib.parse import urlparse
import faiss
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')


def cleaner(text):
    """Replace URLs with format: <url>: (domain/important-path)"""
    if not text:
        return text

    # Regex pattern to match URLs
    url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'

    def replace_url(match):
        url = match.group(0)
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            # Remove www. prefix if present
            if domain.startswith('www.'):
                domain = domain[4:]

            # Extract meaningful path parts (first 1-2 segments)
            path_parts = [part for part in parsed.path.split('/') if part]
            if path_parts:
                # Take first 1-2 meaningful path segments
                important_path = '/'.join(path_parts[:2])
                return f"<url>: ({domain}/{important_path})"
            else:
                return f"<url>: ({domain})"
        except:
            return "<url>: (unknown)"

    return re.sub(url_pattern, replace_url, str(text))


def load_test_data():
    """Load test data."""
    print("Loading test data...")
    test_df = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/test.csv')
    print(f"Loaded {len(test_df)} test examples")
    print(f"Unique rules: {test_df['rule'].nunique()}")
    return test_df


def collect_all_texts(test_df):
    """Collect all unique texts from test set."""
    print("\nCollecting all texts for embedding...")
    
    all_texts = set()
    
    # Add all bodies
    for body in test_df['body']:
        if pd.notna(body):
            all_texts.add(cleaner(str(body)))
    
    # Add all positive and negative examples
    example_cols = ['positive_example_1', 'positive_example_2', 
                   'negative_example_1', 'negative_example_2']
    
    for col in example_cols:
        for example in test_df[col]:
            if pd.notna(example):
                all_texts.add(cleaner(str(example)))
    
    all_texts = list(all_texts)
    print(f"Collected {len(all_texts)} unique texts")
    return all_texts


def generate_embeddings(texts, model, batch_size=64):
    """Generate BGE embeddings for all texts."""
    print(f"Generating embeddings for {len(texts)} texts...")
    
    embeddings = model.encode(
        sentences=texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=False,
        normalize_embeddings=True
    )
    
    return embeddings


def create_test_triplet_dataset(test_df, augmentation_factor=2, random_seed=42, subsample_fraction=1.0):
    """Create triplet dataset from test data: anchor=rule, positive=positive_example, negative=negative_example."""
    random.seed(random_seed)
    np.random.seed(random_seed)
    
    anchors = []
    positives = []
    negatives = []
    
    print("Creating rule-aligned triplets from test data...")
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing test rows"):
        rule = cleaner(str(row['rule']))
        
        pos_examples = []  # Will contain compliant comments (rule-aligned)
        neg_examples = []  # Will contain violating comments (rule-misaligned)

        for neg_col in ['negative_example_1', 'negative_example_2']:  # Compliant → triplet positive
            if pd.notna(row[neg_col]):
                pos_examples.append(cleaner(str(row[neg_col])))

        for pos_col in ['positive_example_1', 'positive_example_2']:  # Violating → triplet negative
            if pd.notna(row[pos_col]):
                neg_examples.append(cleaner(str(row[pos_col])))
        
        for pos_ex in pos_examples:
            for neg_ex in neg_examples:
                anchors.append(rule)
                positives.append(pos_ex)
                negatives.append(neg_ex)
    
    if augmentation_factor > 0:
        print(f"Adding {augmentation_factor}x augmentation...")
        
        rule_positives = {}
        rule_negatives = {}
        
        for rule in test_df['rule'].unique():
            rule_df = test_df[test_df['rule'] == rule]
            
            pos_pool = []
            neg_pool = []
            
            for _, row in rule_df.iterrows():
                for neg_col in ['negative_example_1', 'negative_example_2']:  # Compliant → triplet positive
                    if pd.notna(row[neg_col]):
                        pos_pool.append(cleaner(str(row[neg_col])))
                for pos_col in ['positive_example_1', 'positive_example_2']:  # Violating → triplet negative
                    if pd.notna(row[pos_col]):
                        neg_pool.append(cleaner(str(row[pos_col])))
            
            rule_positives[rule] = list(set(pos_pool))
            rule_negatives[rule] = list(set(neg_pool))
        
        for rule in test_df['rule'].unique():
            clean_rule = cleaner(str(rule))
            pos_pool = rule_positives[rule]
            neg_pool = rule_negatives[rule]
            
            n_samples = min(augmentation_factor * len(pos_pool), len(pos_pool) * len(neg_pool))
            
            for _ in range(n_samples):
                if pos_pool and neg_pool:
                    anchors.append(clean_rule)
                    positives.append(random.choice(pos_pool))
                    negatives.append(random.choice(neg_pool))
    
    combined = list(zip(anchors, positives, negatives))
    random.shuffle(combined)
    
    # Apply subsampling if requested
    original_count = len(combined)
    if subsample_fraction < 1.0:
        n_samples = int(len(combined) * subsample_fraction)
        combined = combined[:n_samples]
        print(f"Subsampled {original_count} -> {len(combined)} triplets ({subsample_fraction*100:.1f}%)")
    
    anchors, positives, negatives = zip(*combined) if combined else ([], [], [])
    
    print(f"Created {len(anchors)} triplets from test data")
    
    dataset = Dataset.from_dict({
        'anchor': list(anchors),
        'positive': list(positives),
        'negative': list(negatives)
    })
    
    return dataset


def fine_tune_model(model, train_dataset, epochs=3, batch_size=32, learning_rate=2e-5, margin=0.25, output_dir="./models/test-finetuned-bge"):
    """Fine-tune the sentence transformer model using triplet loss on test data."""
    
    print(f"Fine-tuning model on {len(train_dataset)} triplets...")
    
    loss = TripletLoss(model=model, triplet_margin=margin)
    
    # Calculate max_steps for small datasets
    dataset_size = len(train_dataset)
    steps_per_epoch = max(1, dataset_size // batch_size)
    max_steps = steps_per_epoch * epochs

    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        warmup_steps=0,
        learning_rate=learning_rate,
        logging_steps=max(1, max_steps // 4),
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        max_grad_norm=1.0,
        dataloader_drop_last=False,
        gradient_checkpointing=True,
        gradient_accumulation_steps = 1,
        max_steps=max_steps,
        report_to="none"
    )
    
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        loss=loss,
    )
    
    trainer.train()
    
    final_model_path = f"{output_dir}/final"
    print(f"Saving fine-tuned model to {final_model_path}...")
    model.save_pretrained(final_model_path)
    
    return model, final_model_path


def load_or_create_finetuned_model(test_df):
    """Load fine-tuned model if exists, otherwise create and fine-tune it."""
    
    fine_tuned_path = "./models/test-finetuned-bge/final"
    
    if os.path.exists(fine_tuned_path):
        print(f"Loading existing fine-tuned model from {fine_tuned_path}...")
        try:
            word_embedding_model = models.Transformer(fine_tuned_path, max_seq_length=128, do_lower_case=True)
            pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
            model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
            print("Loaded fine-tuned model with explicit pooling")
        except:
            model = SentenceTransformer(fine_tuned_path)
            print("Loaded fine-tuned model with default configuration")
        model.half()
        return model
    
    print("Fine-tuned model not found. Creating new one...")
    
    print("Loading base BGE embedding model...")
    try:
        #model_path = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1" 
        #model_path = "/kaggle/input/baai/transformers/bge-base-en-v1.5/1"
        model_path = "/kaggle/input/baai/transformers/bge-base-en-v1.5/1"
        word_embedding_model = models.Transformer(model_path, max_seq_length=128, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from Kaggle path with explicit pooling")
    except:
        model_path = ""  # BAAI/bge-small-en-v1.5
        word_embedding_model = models.Transformer(model_path, max_seq_length=128, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from local path with explicit pooling")
    
    
    triplet_dataset = create_test_triplet_dataset(test_df, augmentation_factor=2, subsample_fraction=1.)

    fine_tuned_model, model_path = fine_tune_model(
        model=base_model,
        train_dataset=triplet_dataset,
        epochs=1,
        batch_size=32,
        learning_rate=2e-5,
        margin=0.25
    )
    
    print(f"Fine-tuning completed. Model saved to: {model_path}")
    fine_tuned_model.half()
    return fine_tuned_model


def generate_rule_embeddings(test_df, model):
    """Generate embeddings for each unique rule."""
    print("Generating rule embeddings...")
    
    unique_rules = test_df['rule'].unique()
    rule_embeddings = {}
    
    for rule in unique_rules:
        clean_rule = cleaner(str(rule))
        rule_emb = model.encode(
            clean_rule,
            convert_to_tensor=False,
            normalize_embeddings=True
        )
        rule_embeddings[rule] = rule_emb
        
    print(f"Generated embeddings for {len(rule_embeddings)} rules")
    return rule_embeddings


def create_rule_centroids(test_df, text_to_embedding, rule_embeddings):
    """Create single centroid (mean) for positive and negative examples for each rule."""
    print(f"\nCreating rule centroids (single mean centroid per type)...")

    rule_centroids = {}

    for rule in test_df['rule'].unique():
        rule_data = test_df[test_df['rule'] == rule]

        # Collect positive examples
        pos_embeddings = []
        for _, row in rule_data.iterrows():
            for col in ['positive_example_1', 'positive_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        pos_embeddings.append(text_to_embedding[clean_text])

        # Collect negative examples
        neg_embeddings = []
        for _, row in rule_data.iterrows():
            for col in ['negative_example_1', 'negative_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        neg_embeddings.append(text_to_embedding[clean_text])

        if pos_embeddings and neg_embeddings:
            pos_embeddings = np.array(pos_embeddings)
            neg_embeddings = np.array(neg_embeddings)

            # Compute mean centroids
            pos_centroid = pos_embeddings.mean(axis=0)
            neg_centroid = neg_embeddings.mean(axis=0)

            # Normalize centroids
            pos_centroid = pos_centroid / np.linalg.norm(pos_centroid)
            neg_centroid = neg_centroid / np.linalg.norm(neg_centroid)

            rule_centroids[rule] = {
                'positive': pos_centroid,
                'negative': neg_centroid,
                'pos_count': len(pos_embeddings),
                'neg_count': len(neg_embeddings),
                'rule_embedding': rule_embeddings[rule]
            }

            print(f"  Rule: {rule[:50]}... - Pos: {len(pos_embeddings)}, Neg: {len(neg_embeddings)}")

    print(f"Created centroids for {len(rule_centroids)} rules")
    return rule_centroids


def predict_test_set(test_df, text_to_embedding, rule_centroids):
    """Predict test set using Euclidean distance between body and pos/neg centroids."""
    print("\nMaking predictions on test set with Euclidean distance...")

    row_ids = []
    predictions = []

    for rule in test_df['rule'].unique():
        print(f"  Processing rule: {rule[:50]}...")
        # Select entries from test following rule
        rule_data = test_df[test_df['rule'] == rule] 

        if rule not in rule_centroids:
            continue

        # Load centroids from rule
        pos_centroid = rule_centroids[rule]['positive']
        neg_centroid = rule_centroids[rule]['negative']

        # Collect all valid embeddings and row_ids for this rule
        valid_embeddings = []
        valid_row_ids = []
        for _, row in rule_data.iterrows():
            body = cleaner(str(row['body']))
            row_id = row['row_id']

            if body in text_to_embedding:
                valid_embeddings.append(text_to_embedding[body])
                valid_row_ids.append(row_id)

        if not valid_embeddings:
            continue

        # Convert to numpy array
        query_embeddings = np.array(valid_embeddings)

        # Compute Euclidean distances
        pos_distances = np.linalg.norm(query_embeddings - pos_centroid, axis=1)
        neg_distances = np.linalg.norm(query_embeddings - neg_centroid, axis=1)

        # Score: closer to positive (lower distance) = higher violation score
        rule_predictions = neg_distances - pos_distances

        row_ids.extend(valid_row_ids)
        predictions.extend(rule_predictions)

    print(f"Made predictions for {len(predictions)} test examples")
    return row_ids, np.array(predictions)




def main():
    """Main inference pipeline."""
    print("="*70)
    print("SIMPLE SIMILARITY CLASSIFIER - INFERENCE")
    print("="*70)
    
    # Step 1: Load test data
    test_df = load_test_data()

    # Step 1.5: Add train data
    train_df = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/train.csv')
    for i, row in train_df.iterrows():
        # if rule_violation is true, replace positive_example_1 with body, else replace negative_example_1 with body
        if row.rule_violation:
            train_df.at[i, 'positive_example_1'] = row.body
        else:
            train_df.at[i, 'negative_example_1'] = row.body
    
    # stack both dataframes
    train_df.drop(columns=['rule_violation'], inplace=True)
    test_df_plus = pd.concat([test_df, train_df], ignore_index=True)
    
    # Step 2: Load or create fine-tuned model
    print("\n" + "="*50)
    print("MODEL PREPARATION PHASE")
    print("="*50)
    model = load_or_create_finetuned_model(test_df_plus)      # Uses rule + pos + negs
    
    # Step 3: Collect all texts
    all_texts = collect_all_texts(test_df_plus)               # Uses body + pos + negs
    # Step 4: Generate embeddings with fine-tuned model
    print("\n" + "="*50)
    print("EMBEDDING GENERATION PHASE")
    print("="*50)
    all_embeddings = generate_embeddings(all_texts, model)  
    # Step 5: Create text to embedding mapping
    text_to_embedding = {text: emb for text, emb in zip(all_texts, all_embeddings)}
    
    # Step 6: Generate rule embeddings
    rule_embeddings = generate_rule_embeddings(test_df, model)   # Uses rule
    
    # Step 7: Create rule centroids from test examples
    rule_centroids = create_rule_centroids(test_df_plus, text_to_embedding, rule_embeddings)  # Uses rule + pos + negs
    
    # Step 8: Predict test set
    print("\n" + "="*50)
    print("PREDICTION PHASE")
    print("="*50)
    row_ids, predictions = predict_test_set(test_df, text_to_embedding, rule_centroids) # Uses body
    
    # Step 9: Create submission with rule-conditioned scores
    submission_df = pd.DataFrame({
        'row_id': row_ids,
        'rule': [test_df[test_df['row_id'] == rid]['rule'].iloc[0] for rid in row_ids],
        'rule_violation': predictions
    })
    
    submission_df.to_csv('submission_faiss_triplet_0_909.csv', index=False)
if __name__ == "__main__":
    main()

Writing triplet_faiss_public.py


# Deberta-Nli  

Natural Language Inference (NLI) with test-time adaptation — treat rule violation detection as an entailment problem and fine-tune a DeBERTa model on each rule's canonical examples at test time.

Base Model: `MoritzLaurer/deberta-v3-large-zeroshot-v2.0`


1. NLI Reformulation
- Premise: comment body text
- Hypothesis: "This comment violates the rule: {rule}" (8 paraphrases)
- Label: Entailment = violation, Contradiction/Non-entailment = compliant



2. Test-Time Training (TTT) per Rule
- Extract canonical positive/negative examples from test CSV
- Upsample them with multipliers to reach ~25k target pairs
- Train on this rule-specific mini-dataset for 2 epochs
- Minimal scope: only update biases + classifier head + LayerNorms (frozen backbone)


4. Data Balancing
- Compute dynamic multipliers for pos/neg to achieve ~50/50 class balance
- Capped at 32x to prevent overfitting
- Optional dual-hypothesis: also include "does not violate" paraphrases (flipped labels)
    

5. Scoring
- Average predictions across 8 "violates" paraphrases (ensemble)
- For binary heads: use entailment probability directly
- For 3-way heads: normalize as entailment / (entailment + contradiction + 1e-12)

In [4]:
%%writefile deberta_nli_zeroshot.py

# 0.906
import os, gc, math, random
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List
from tqdm import tqdm
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = True

MODEL_NAME   = "/kaggle/input/download-models-jigsaw/deberta-v3-base-zeroshot-v2.0"

TEST_PATH    = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
SUB_PATH     = "submission_deberta_nli_0_906.csv"

SCOPE        = "bias+head+ln"   # train biases + classifier head + LayerNorms

LR            = 7e-5
EPOCHS        = 2
LR_SCHED      = "cosine"
TARGET_PAIRS  = 25_000
POOLING       = "mean"          # over 'violates' paraphrases
MAX_LENGTH    = 448
BATCH_SIZE    = 16
WARMUP_RATIO  = 0.02
WEIGHT_DECAY  = 0.0
GRAD_ACCUM    = 1
MAX_MULT      = 32

# boosters
BALANCE_CLASSES          = True
USE_DUAL_HYP             = False   # include the "does not violate" hypothesis family
AUX_NEG_FROM_OTHER_RULES = True   # add small fraction of other-rule bodies as negatives
AUX_NEG_FRACTION         = 0.10
# -------------------------------------------------------------------

PARA_VIOLATE = [
    "This comment violates the rule: {rule}.",
    "The comment breaks the rule: {rule}.",
    "This text is in violation of: {rule}.",
    "The content contravenes the rule: {rule}.",
    "It is true that this comment violates the rule: {rule}.",
    "This comment is non-compliant with: {rule}.",
    "The comment infringes the rule: {rule}.",
    "This content is disallowed under: {rule}.",
]
PARA_NONVIOLATE = [
    "This comment does not violate the rule: {rule}.",
    "The comment complies with the rule: {rule}.",
    "This text does not break the rule: {rule}.",
    "The content is allowed under the rule: {rule}.",
    "It is false that this comment violates the rule: {rule}.",
    "This comment is compliant with: {rule}.",
    "The comment does not infringe the rule: {rule}.",
    "This content is permitted under: {rule}.",
]

# ---------------- data ----------------
df = pd.read_csv(TEST_PATH)
df["rule"] = df["rule"].fillna("").astype(str)
df["body"] = df["body"].fillna("").astype(str)
rules_unique = sorted(df["rule"].unique().tolist())
print(f"Found rules: {len(rules_unique)}")


id_col = "row_id"

ex_pos_cols = [c for c in df.columns if "positive_example" in c]
ex_neg_cols = [c for c in df.columns if "negative_example" in c]

def collect_canonical_examples(rule_name: str):
    r_df = df[df["rule"] == rule_name]
    pos, neg = [], []
    for c in ex_pos_cols: pos += r_df[c].dropna().astype(str).tolist()
    for c in ex_neg_cols: neg += r_df[c].dropna().astype(str).tolist()
    # dedupe + strip
    pos = list(dict.fromkeys([s.strip() for s in pos if s and s.strip()]))
    neg = list(dict.fromkeys([s.strip() for s in neg if s and s.strip()]))
    # decontaminate overlaps
    sp, sn = set(pos), set(neg)
    pos = [x for x in pos if x not in sn]
    neg = [x for x in neg if x not in sp]
    return pos, neg

def maybe_add_aux_negs(neg_list: List[str], rule: str, base_total: int):
    if not AUX_NEG_FROM_OTHER_RULES or base_total == 0:
        return neg_list
    cap = max(0, int(AUX_NEG_FRACTION * base_total))
    if cap == 0: 
        return neg_list
    # sample bodies from other rules (test-time training)
    other = df[df["rule"] != rule]["body"].tolist()
    random.shuffle(other)
    return neg_list + other[:cap]

def compute_multipliers(n_pos, n_neg, target_total, dual=USE_DUAL_HYP, balance=BALANCE_CLASSES, max_mult=MAX_MULT):
    family_total = max(1, target_total // (2 if dual else 1))
    if balance and n_pos > 0 and n_neg > 0:
        pos_mult = min(max_mult, max(1, math.ceil((0.5 * family_total) / n_pos)))
        neg_mult = min(max_mult, max(1, math.ceil((0.5 * family_total) / n_neg)))
    else:
        base = max(1, n_pos + n_neg)
        m = min(max_mult, max(1, math.ceil(family_total / base)))
        pos_mult = neg_mult = m
    return pos_mult, neg_mult

# ---------------- NLI utils ----------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def detect_label_indices(cfg):
    """Support both 3-way NLI (entailment/contradiction/neutral) and 2-way (entailment/not_entailment)."""
    id2label = {int(k): v for k, v in getattr(cfg, "id2label", {}).items()}
    label2id = {k.lower(): int(v) for k, v in getattr(cfg, "label2id", {}).items()}

    def find_idx(name_set):
        for i, lab in id2label.items():
            if lab.lower().replace("-", "_") in name_set:
                return i
        for nm in name_set:
            nm2 = nm.replace("-", "_")
            if nm2 in label2id:
                return label2id[nm2]
        return None

    ent = find_idx({"entailment"})
    con = find_idx({"contradiction", "not_entailment", "non_entailment", "notentailment"})
    neu = find_idx({"neutral"})

    if ent is None:
        raise RuntimeError(f"Could not find 'entailment' index. id2label={id2label}")
    if con is None:
        con = find_idx({"negative", "other"})
    if con is None:
        raise RuntimeError(f"NLI negative label not found (expected 'contradiction' or 'not_entailment'). id2label={id2label}")

    is_binary = (neu is None)
    return ent, con, is_binary

def make_model(scope: str):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
    # freeze all
    for p in model.parameters(): 
        p.requires_grad = False
    # biases
    if scope in ("bias", "bias+head+ln", "bias+head", "bias+ln"):
        for n, p in model.named_parameters():
            if "bias" in n: 
                p.requires_grad = True
    # classifier head
    if scope in ("bias+head+ln", "bias+head", "head-only"):
        for n, p in model.named_parameters():
            if ("classifier" in n or "score" in n):
                p.requires_grad = True
    # LayerNorms
    if scope in ("bias+head+ln", "bias+ln"):
        for n, p in model.named_parameters():
            if ("LayerNorm" in n or "layer_norm" in n):
                p.requires_grad = True

    ent, neg, is_binary = detect_label_indices(model.config)
    return model, ent, neg, is_binary

@dataclass
class NLISimpleDataset:
    premises: List[str]
    hypotheses: List[str]
    labels: List[int]
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {"premise": self.premises[idx], "hypothesis": self.hypotheses[idx], "labels": int(self.labels[idx])}

class NLIPairCollator:
    def __init__(self, tok, max_length=MAX_LENGTH):
        self.tok = tok; self.max_length = max_length; self.inner = DataCollatorWithPadding(self.tok)
    def __call__(self, features):
        enc = self.tok(
            [f["premise"] for f in features],
            [f["hypothesis"] for f in features],
            truncation=True, padding=True, max_length=self.max_length, return_tensors="pt"
        )
        enc["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return enc

def build_family(prem_list, hyps, label_id):
    H = max(1, len(hyps))
    hyp = [hyps[i % H] for i in range(len(prem_list))]
    lab = [label_id] * len(prem_list)
    return hyp, lab

def adapt_for_rule(rule: str):
    # canonical examples for the rule (from test.csv)
    pos_ex, neg_ex = collect_canonical_examples(rule)
    base_total = len(pos_ex) + len(neg_ex)
    if AUX_NEG_FROM_OTHER_RULES and base_total > 0:
        neg_ex = maybe_add_aux_negs(neg_ex, rule, base_total)
    if len(pos_ex) == 0 or len(neg_ex) == 0:
        print(f"⚠️  Rule '{rule[:60]}' missing examples; skipping adaptation (zero-shot).")
        return None, None, None, None

    pos_mult, neg_mult = compute_multipliers(len(pos_ex), len(neg_ex), TARGET_PAIRS)
    pos_up = pos_ex * pos_mult
    neg_up = neg_ex * neg_mult

    violates_hyps   = [p.format(rule=rule) for p in PARA_VIOLATE]
    nonviol_hyps    = [p.format(rule=rule) for p in PARA_NONVIOLATE] if USE_DUAL_HYP else []

    model, ENT_IDX, NEG_IDX, IS_BINARY = make_model(SCOPE)
    model.train()

    # Family A: "violates" → pos->ENT, neg->NEG
    hyp_A_pos, lab_A_pos = build_family(pos_up, violates_hyps, ENT_IDX)
    hyp_A_neg, lab_A_neg = build_family(neg_up, violates_hyps, NEG_IDX)

    premises   = pos_up + neg_up
    hypotheses = hyp_A_pos + hyp_A_neg
    labels     = lab_A_pos + lab_A_neg

    # Family B: "does not violate" → pos->NEG, neg->ENT
    if USE_DUAL_HYP:
        hyp_B_pos, lab_B_pos = build_family(pos_up, nonviol_hyps, NEG_IDX)
        hyp_B_neg, lab_B_neg = build_family(neg_up, nonviol_hyps, ENT_IDX)
        premises   += pos_up + neg_up
        hypotheses += hyp_B_pos + hyp_B_neg
        labels     += lab_B_pos + lab_B_neg

    # trim to target
    if len(premises) > TARGET_PAIRS:
        rng = np.random.RandomState(SEED)
        idx = rng.permutation(len(premises))[:TARGET_PAIRS]
        premises   = [premises[i] for i in idx]
        hypotheses = [hypotheses[i] for i in idx]
        labels     = [labels[i] for i in idx]

    print(f"🔧 Adapt '{rule[:68]}': pos={len(pos_ex)}, neg={len(neg_ex)}, "
          f"pos_mult={pos_mult}, neg_mult={neg_mult}, rows={len(premises)}")

    train_ds = NLISimpleDataset(premises, hypotheses, labels)

    args = TrainingArguments(
        output_dir=f"deberta_ttt_{abs(hash(rule))%10_000}",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        gradient_accumulation_steps=GRAD_ACCUM,
        lr_scheduler_type=LR_SCHED,
        logging_steps=500,
        save_strategy="no",
        eval_strategy="no",
        report_to=[],
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
        seed=SEED,
        remove_unused_columns=False,
        max_grad_norm=1.0,
    )
    collator = NLIPairCollator(tokenizer, MAX_LENGTH)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator, tokenizer=tokenizer)
    trainer.train()
    del trainer; gc.collect(); torch.cuda.empty_cache()
    return model, ENT_IDX, NEG_IDX, IS_BINARY

def score_rule(model, ENT_IDX, NEG_IDX, texts: List[str], rule: str, IS_BINARY: bool):
    """Score 'violation' with paraphrases; binary heads use entailment prob directly;
       3-way heads use pe / (pe + pc)."""
    hyps = [p.format(rule=rule) for p in PARA_VIOLATE]
    model.eval()
    all_probs = []
    with torch.no_grad():
        for h in hyps:
            probs_yes = []
            for i in range(0, len(texts), BATCH_SIZE):
                enc = tokenizer(
                    texts[i:i+BATCH_SIZE], [h]*min(BATCH_SIZE, len(texts)-i),
                    truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt"
                ).to(DEVICE)
                logits = model(**enc).logits
                p = torch.softmax(logits, dim=-1)
                if IS_BINARY:
                    pe = p[:, ENT_IDX]
                    probs = pe  # binary: direct entailment prob
                else:
                    pe, pc = p[:, ENT_IDX], p[:, NEG_IDX]   # 3-way: NEG_IDX is contradiction
                    probs = pe / (pe + pc + 1e-12)
                probs_yes.append(probs.detach().float().cpu().numpy())
            all_probs.append(np.concatenate(probs_yes))
    stacked = np.stack(all_probs, axis=1)
    return stacked.mean(axis=1) if POOLING == "mean" else stacked.max(axis=1)

# ---------------- SUBMISSION ----------------
def generate_submission():
    preds = np.zeros(len(df), dtype=np.float32)

    for i, rule in enumerate(rules_unique, 1):
        print("\n" + "="*100)
        print(f"[{i}/{len(rules_unique)}] Rule: {rule}")
        # Adapt for this rule using test canonicals (TTT)
        model, ENT_IDX, NEG_IDX, IS_BINARY = adapt_for_rule(rule)
        if model is None:  # zero-shot fallback if missing canonicals
            model, ENT_IDX, NEG_IDX, IS_BINARY = make_model(SCOPE)

        # Score test bodies for this rule
        mask = (df["rule"] == rule).values
        texts = df.loc[mask, "body"].tolist()
        print(f"Scoring {len(texts)} bodies...")
        p = score_rule(model, ENT_IDX, NEG_IDX, texts, rule, IS_BINARY)
        preds[mask] = p.astype(np.float32)

        # free
        del model; gc.collect(); torch.cuda.empty_cache()


    out = pd.DataFrame({
        "row_id": df["row_id"].values,
        "rule": df["rule"].astype(str).values,
        "rule_violation": np.clip(preds, 1e-6, 1 - 1e-6),
    })

    out.to_csv(SUB_PATH, index=False)
    print(f"\n✅ Saved {SUB_PATH} | mean={out['rule_violation'].mean():.4f}  "
          f"rng=[{out['rule_violation'].min():.4f},{out['rule_violation'].max():.4f}]  "
          f"shape={out.shape}")
    return out

sub = generate_submission()

Writing deberta_nli_zeroshot.py


# Qwen 2.5 32b with Asymmetric Example Selection
Similarity-based example selection — use embedding similarity to dynamically choose the most relevant canonical examples for each comment before passing to the LLM. Qwen 2.5 32b trained on train set.

Asymmetric Selection: Rank examples by similarity and select asymmetrically:

- Positive examples: Pick the MOST similar (clear violation patterns)
- Negative examples: Pick the LEAST similar (diverse boundary cases)

This sharpens the decision boundary by showing the model clear violations alongside hard negatives.

### Pipeline

1. Embed Rule Examples (Qwen3 0.6B embedding model)
- Encode 2 positive + 2 negative canonical examples per rule

2. Dynamic Selection per Comment
- Encode comment + rule as query
- Calculate cosine similarities to all examples
- Select best positive (argmax), worst negative (argmin)
- Include quality thresholds to avoid poor selections

3. Construct Prompts with selected examples

4. LLM Inference (Qwen 32B-Instruct GPTQ)
- Fine-tuned with LoRA checkpoint
- Multiple choice processor constrained to "Yes"/"No"
- Extract logits, apply softmax

Results
Score: 0.890 -> +0.006 improvement over vanilla LLM baseline, demonstrating that intelligent example ordering significantly boosts LLM classification quality.

Why It Works:
- Few-shot learning improves when examples are contextually relevant (most similar positives)
- Hard negatives (least similar) force the model to learn sharper decision boundaries
- Asymmetry mirrors human reasoning: focus on prototype violations, edge cases for compliance

In [5]:
%%writefile qwen_32b_inference.py

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import torch.nn.functional as F
from tqdm import tqdm
import time
import torch, vllm
from vllm.lora.request import LoRARequest
import os
import pandas as pd
import gc
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from scipy.special import softmax

# Configuration
MODEL_NAME = "/kaggle/input/qwen2.5/transformers/32b-instruct-gptq-int4/1"
LORA_PATH = "/kaggle/input/qwen-32b-llm-ranked-balanced-data/checkpoint-420"
os.environ["VLLM_USE_V1"] = "0"


# System prompt
SYS_PROMPT = (
    "You are an expert reddit content moderator. "
    "Compare the comment to the rule using the examples. "
    "Respond only Yes or No."
)

class Qwen3AsymmetricSelector:
    """
    Strategy A: Asymmetric example selection
    - MOST similar positive examples (clear violation patterns)
    - LEAST similar negative examples (diverse boundary cases)
    """
    
    def __init__(self):
        print("Loading Qwen3 embedding model for Strategy A...")
        
        import kagglehub
        model_dir = kagglehub.model_download("qwen-lm/qwen-3-embedding/transformers/0.6b")
        
        # Load model using official Qwen3 approach
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir, padding_side='left')
        
        try:
            # Try flash attention first
            self.model = AutoModel.from_pretrained(
                model_dir, 
                attn_implementation="flash_attention_2", 
                torch_dtype=torch.float16
            ).cuda()
            print("Using flash attention")
        except:
            # Fall back to standard attention
            self.model = AutoModel.from_pretrained(
                model_dir,
                torch_dtype=torch.float16,
                device_map="auto"
            )
            print("Using standard attention")
            
        self.model.eval()
        self.device = next(self.model.parameters()).device
        print(f"Qwen3 embedding model loaded on {self.device}")
    
    def last_token_pool(self, last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        """Official Qwen3 pooling method"""
        left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
        if left_padding:
            return last_hidden_states[:, -1]
        else:
            sequence_lengths = attention_mask.sum(dim=1) - 1
            batch_size = last_hidden_states.shape[0]
            return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]
    
    def get_detailed_instruct(self, task_description: str, query: str) -> str:
        """Official Qwen3 instruction format"""
        return f'Instruct: {task_description}\nQuery: {query}'
    
    def encode_documents(self, texts, batch_size=16, max_length=2048):
        """Encode documents (examples) without instruction format"""
        all_embeddings = []
        
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            
            batch_dict = self.tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            batch_dict = {k: v.to(self.device) for k, v in batch_dict.items()}
            
            with torch.no_grad():
                outputs = self.model(**batch_dict)
                embeddings = self.last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
                embeddings = F.normalize(embeddings, p=2, dim=1)
                all_embeddings.append(embeddings.cpu())
        
        return torch.cat(all_embeddings, dim=0)

    def encode_query(self, query_text, task_description, max_length=2048):
        """Encode a single query with instruction format"""
        query_with_instruction = self.get_detailed_instruct(task_description, query_text)
        
        batch_dict = self.tokenizer(
            [query_with_instruction],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        batch_dict = {k: v.to(self.device) for k, v in batch_dict.items()}
        
        with torch.no_grad():
            outputs = self.model(**batch_dict)
            embedding = self.last_token_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
            embedding = F.normalize(embedding, p=2, dim=1)
        
        return embedding

    def precompute_rule_examples(self, df):
        """Precompute embeddings for all rule examples"""
        print("Precomputing rule example embeddings with Qwen3...")
        
        rule_examples = {}
        
        for rule in tqdm(df['rule'].unique(), desc="Processing rules"):
            rule_df = df[df['rule'] == rule].iloc[0]
            
            # Create enhanced context for examples
            pos_examples = [
                f"Rule: {rule}\nViolation example: {rule_df['positive_example_1']}",
                f"Rule: {rule}\nViolation example: {rule_df['positive_example_2']}"
            ]
            neg_examples = [
                f"Rule: {rule}\nCompliant example: {rule_df['negative_example_1']}",
                f"Rule: {rule}\nCompliant example: {rule_df['negative_example_2']}"
            ]
            
            # Encode examples as documents
            pos_embeddings = self.encode_documents(pos_examples)
            neg_embeddings = self.encode_documents(neg_examples)
            
            rule_examples[rule] = {
                'positive_examples': [rule_df['positive_example_1'], rule_df['positive_example_2']],
                'negative_examples': [rule_df['negative_example_1'], rule_df['negative_example_2']],
                'pos_embeddings': pos_embeddings,
                'neg_embeddings': neg_embeddings
            }
        
        return rule_examples

    def select_asymmetric_examples(self, comment, rule, rule_examples):
        """
        Strategy A: Asymmetric selection
        - MAXIMIZE positive similarity (clear violation patterns)
        - MINIMIZE negative similarity (diverse boundary cases)
        """
        task_description = "Given a content moderation rule and comment, find the most similar violation and diverse compliance examples"
        
        query_text = f"Rule: {rule}\nComment to evaluate: {comment}"
        query_embedding = self.encode_query(query_text, task_description)
        
        rule_data = rule_examples[rule]
        
        # Move embeddings to same device
        pos_embeddings = rule_data['pos_embeddings'].to(query_embedding.device)
        neg_embeddings = rule_data['neg_embeddings'].to(query_embedding.device)
        
        # Calculate similarities
        pos_similarities = (query_embedding @ pos_embeddings.T).squeeze().cpu().numpy()
        neg_similarities = (query_embedding @ neg_embeddings.T).squeeze().cpu().numpy()
        
        # Handle single embedding case
        if pos_similarities.ndim == 0:
            pos_similarities = np.array([pos_similarities])
        if neg_similarities.ndim == 0:
            neg_similarities = np.array([neg_similarities])
        
        # STRATEGY A: ASYMMETRIC SELECTION
        # Positive: MOST similar (clear violation patterns)
        best_pos_idx = np.argmax(pos_similarities)
        remaining_pos_idx = 1 - best_pos_idx
        
        # Negative: LEAST similar (diverse boundary cases)
        best_neg_idx = np.argmin(neg_similarities)  # KEY CHANGE: argmin instead of argmax
        remaining_neg_idx = 1 - best_neg_idx
        
        # Quality checks with different thresholds
        pos_threshold = 0.2  # Minimum threshold for positive similarity
        neg_threshold = 0.8  # Maximum threshold for negative similarity (we want dissimilar)
        
        # If positive similarity is too low, use original order
        if pos_similarities[best_pos_idx] < pos_threshold:
            best_pos_idx, remaining_pos_idx = 0, 1
        
        # If negative similarity is too high (too similar), use original order
        if neg_similarities[best_neg_idx] > neg_threshold:
            best_neg_idx, remaining_neg_idx = 0, 1
        
        selected_examples = {
            'positive_example_1': rule_data['positive_examples'][best_pos_idx],
            'positive_example_2': rule_data['positive_examples'][remaining_pos_idx],
            'negative_example_1': rule_data['negative_examples'][best_neg_idx],
            'negative_example_2': rule_data['negative_examples'][remaining_neg_idx],
            'pos_similarity_max': float(pos_similarities[best_pos_idx]),
            'neg_similarity_min': float(neg_similarities[best_neg_idx]),
            'strategy': 'asymmetric_A'
        }
        
        return selected_examples

def create_asymmetric_prompts_memory_safe(df):
    """
    Create prompts using Strategy A - MEMORY SAFE VERSION
    """
    print("=== STRATEGY A: Asymmetric Example Selection ===")
    
    # Initialize selector
    selector = Qwen3AsymmetricSelector()
    
    # Precompute embeddings
    rule_examples = selector.precompute_rule_examples(df)
    
    # Generate optimized prompts
    prompt_texts = []
    similarity_scores = []
    
    print("Selecting asymmetric examples for each comment...")
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Optimizing examples"):
        selected_examples = selector.select_asymmetric_examples(
            row.body, row.rule, rule_examples
        )
        
        # Store analysis data
        similarity_scores.append({
            'row_id': row.get('row_id', i),
            'pos_similarity_max': selected_examples['pos_similarity_max'],
            'neg_similarity_min': selected_examples['neg_similarity_min'],
            'strategy': selected_examples['strategy']
        })
        
        # Create prompt with asymmetrically selected examples
        text = f"""
Rule: {row.rule}
Examples that DO NOT violate this rule:
1. {selected_examples['negative_example_1']}\n
2. {selected_examples['negative_example_2']}\n
Examples that VIOLATE this rule:
1. {selected_examples['positive_example_1']}\n
2. {selected_examples['positive_example_2']}\n
Comment:\n{row.body}
"""
        prompt_texts.append(text)
    
    # CRITICAL: Clear Qwen3 from memory
    print("Clearing Qwen3 model from GPU memory...")
    del selector.model
    del selector.tokenizer  
    del selector
    del rule_examples
    
    # Aggressive memory clearing
    for _ in range(3):
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    
    torch.cuda.synchronize()
    time.sleep(5)  # Give system time to fully clear
    
    print(f"✅ Strategy A completed. Generated {len(prompt_texts)} asymmetric prompts")
    
    return prompt_texts, similarity_scores

def run_strategy_a_inference(df):
    """
    Complete Strategy A inference pipeline
    """
    print("="*80)
    print("STRATEGY A: ASYMMETRIC EXAMPLE SELECTION")
    print("- Maximize positive similarity (clear violations)")
    print("- Minimize negative similarity (diverse boundaries)")
    print("="*80)
    
    # Phase 1: Generate asymmetric prompts and clear memory
    prompt_texts, similarity_scores = create_asymmetric_prompts_memory_safe(df)
    
    # Phase 2: Initialize VLLM
    print("\n=== PHASE 2: Initializing VLLM ===")
    
    llm = vllm.LLM(
        MODEL_NAME,
        quantization='gptq',
        tensor_parallel_size=2,  # Conservative for stability
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=1920,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
    )
    
    tokenizer = llm.get_tokenizer()
    print("VLLM initialized successfully")
    
    # Phase 3: Apply chat templates
    print("\n=== PHASE 3: Creating final prompts ===")
    final_prompts = []
    
    for text in tqdm(prompt_texts, desc="Applying chat templates"):
        messages = [
            {"role": "system", "content": SYS_PROMPT},
            {"role": "user", "content": text}
        ]
        
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        ) + "Answer:"
        
        final_prompts.append(prompt)
    
    # Phase 4: Run VLLM inference
    print("\n=== PHASE 4: Running VLLM inference ===")
    
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=['Yes','No'])
    
    outputs = llm.generate(
        final_prompts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )
    
    # Phase 5: Process results
    print("\n=== PHASE 5: Processing results ===")
    
    logprobs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    
    logit_matrix = pd.DataFrame(logprobs)[['Yes','No']]
    df_final = pd.concat([df, logit_matrix], axis=1)
    df_final[['Yes',"No"]] = df_final[['Yes',"No"]].apply(lambda x: softmax(x.values), axis=1, result_type="expand")
    df_final["pred"] = df_final["Yes"]
    df_final['rule_violation'] = df_final["pred"]
    
    # Save results
    df_final[['row_id', 'rule', 'rule_violation']].to_csv("submission_qwen32b_0_896.csv", index=False)
     
    
    return df_final

if __name__ == '__main__':
    # Load data
    df = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv") 
    # Run Strategy A inference
    df_result = run_strategy_a_inference(df)
    
    print(df_result.head())

Writing qwen_32b_inference.py


# Public Deberta
- Increase lr
- No warmup

In [6]:
%%writefile utils.py

import pandas as pd
import re

def url_to_semantics(text: str) -> str:
    if not isinstance(text, str):
        return ""

    url_pattern = r'https?://[^\s/$.?#].[^\s]*'
    urls = re.findall(url_pattern, text)
    
    if not urls:
        return "" 

    all_semantics = []
    seen_semantics = set()

    for url in urls:
        url_lower = url.lower()
        
        domain_match = re.search(r"(?:https?://)?([a-z0-9\-\.]+)\.[a-z]{2,}", url_lower)
        if domain_match:
            full_domain = domain_match.group(1)
            parts = full_domain.split('.')
            for part in parts:
                if part and part not in seen_semantics and len(part) > 3: # Avoid short parts like 'www'
                    all_semantics.append(f"domain:{part}")
                    seen_semantics.add(part)

        # 2. Extract path parts
        path = re.sub(r"^(?:https?://)?[a-z0-9\.-]+\.[a-z]{2,}/?", "", url_lower)
        path_parts = [p for p in re.split(r'[/_.-]+', path) if p and p.isalnum()] # Split by common delimiters

        for part in path_parts:
            # Clean up potential file extensions or query params
            part_clean = re.sub(r"\.(html?|php|asp|jsp)$|#.*|\?.*", "", part)
            if part_clean and part_clean not in seen_semantics and len(part_clean) > 3:
                all_semantics.append(f"path:{part_clean}")
                seen_semantics.add(part_clean)

    if not all_semantics:
        return ""

    return f"\nURL Keywords: {' '.join(all_semantics)}"


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv") 
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []

    flatten.append(train_dataset[["body", "rule", "subreddit","rule_violation"]].copy())

    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"
            
            if col_name in train_dataset.columns:
                sub_dataset = train_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
                
                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]
                
                if not sub_dataset.empty:
                    flatten.append(sub_dataset)
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col_name = f"{violation_type}_example_{i}"
            
            if col_name in test_dataset.columns:
                sub_dataset = test_dataset[[col_name, "rule", "subreddit"]].copy()
                sub_dataset = sub_dataset.rename(columns={col_name: "body"})
                sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
                
                sub_dataset.dropna(subset=['body'], inplace=True)
                sub_dataset = sub_dataset[sub_dataset['body'].str.strip().str.len() > 0]
                
                if not sub_dataset.empty:
                    flatten.append(sub_dataset)
    
    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(subset=['body', 'rule', 'subreddit'], ignore_index=True)
    dataframe.drop_duplicates(subset=['body','rule'],keep='first',inplace=True)
    
    return dataframe.sample(frac=1, random_state=42).reset_index(drop=True)

Writing utils.py


In [7]:
%%writefile train_deberta.py

import os
import pandas as pd
import torch
import random
import numpy as np
from sklearn.model_selection import train_test_split 
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from utils import get_dataframe_to_train, url_to_semantics

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class CFG:
    model_name_or_path = "/kaggle/input/download-models-jigsaw/deberta-v3-base-zeroshot-v2.0"
    data_path = "/kaggle/input/jigsaw-agile-community-rules/"
    output_dir = "./deberta_v3_small_final_model"
  
    EPOCHS = 3
    LEARNING_RATE = 3e-5  
    
    MAX_LENGTH = 512
    BATCH_SIZE = 8

class JigsawDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

def main():
    seed_everything(42)
    training_data_df = get_dataframe_to_train(CFG.data_path)
    # training_data_df, valid_df = train_test_split(full_df,test_size=0.2,stratify=full_df['rule'],random_state=42)
    print(f"Training dataset (from examples only) size: {len(training_data_df)}")

    test_df_for_prediction = pd.read_csv(f"{CFG.data_path}/test.csv")
    
    training_data_df['body_with_url'] = training_data_df['body'].apply(lambda x: x + url_to_semantics(x))
    training_data_df['input_text'] = training_data_df['rule'] + "[SEP]" + training_data_df['body_with_url']

    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name_or_path)
    train_encodings = tokenizer(training_data_df['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    train_labels = training_data_df['rule_violation'].tolist()
    train_dataset = JigsawDataset(train_encodings, train_labels)

    model = AutoModelForSequenceClassification.from_pretrained(CFG.model_name_or_path, num_labels=2)
    
    training_args = TrainingArguments(
        output_dir=CFG.output_dir,
        num_train_epochs=CFG.EPOCHS,
        learning_rate=CFG.LEARNING_RATE,
        per_device_train_batch_size=CFG.BATCH_SIZE,
        warmup_ratio=0.001,
        weight_decay=0.01,
        report_to="none",
        save_strategy="no",  #这一行加上这个 save_strategy="no"
        logging_steps=1,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )
    
    trainer.train()

    test_df_for_prediction['body_with_url'] = test_df_for_prediction['body'].apply(lambda x: x + url_to_semantics(x))
    test_df_for_prediction['input_text'] = test_df_for_prediction['rule'] + "[SEP]" + test_df_for_prediction['body_with_url']
    
    test_encodings = tokenizer(test_df_for_prediction['input_text'].tolist(), truncation=True, padding=True, max_length=CFG.MAX_LENGTH)
    test_dataset = JigsawDataset(test_encodings)
    
    predictions = trainer.predict(test_dataset)
    probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=1)[:, 1].numpy()

    submission_df = pd.DataFrame({
        "row_id": test_df_for_prediction["row_id"],
       "rule": test_df_for_prediction["rule"],  # Added this line
        "rule_violation": probs
    })
    submission_df.to_csv("submission_public_0_906.csv", index=False)

if __name__ == "__main__":
    main()

Writing train_deberta.py


# Qwen3 8b DDP
Train and infer with max len 512. 
Remove duplicates. 
Use sft DDP. 

Training and inference in ~2hrs. 

In [8]:
%%writefile train_qwen8b.py

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers.utils import is_torch_bf16_gpu_available
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    LogitsProcessor,
    BitsAndBytesConfig
)
import pandas as pd
from datasets import Dataset
import os 

import pandas as pd
from datasets import Dataset
import random, numpy as np
random.seed(42)
np.random.seed(42)

# -----------------------
# Global configuration
# -----------------------
BASE_MODEL_PATH = "/kaggle/input/qwen-3/transformers/8b/1"
LORA_PATH = "output/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"
MAX_SAMPLES = 30000
TEST_SAMPLE_FRAC = 1.0
TEST_SAMPLE_SEED = 42

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = (
    "You are a expert Reddit Moderator, you are given a comment and a rule. "
    "Your task is to classify whether the comment violates the rule. Only respond Yes/No."
)


def build_prompt(row):
    return f"<|im_start|>system\n{BASE_PROMPT}<|im_end|>\n<|im_start|>user\n<Rule>{row['rule']}</Rule>\n<Comment>{row['body']}</Comment><|im_end|>\n<|im_start|>assistant\n{COMPLETE_PHRASE}"


def get_dataframe_to_train(data_path):
    # If Kaggle "public" 10-row test.csv, downsample train a bit for speed.
    if len(pd.read_csv(f"{data_path}/test.csv")) == 10:
        train_frac = 0.1
    else:
        train_frac = 1.0

    train_dataset = pd.read_csv(f"{data_path}/train.csv").sample(
        frac=train_frac, random_state=TEST_SAMPLE_SEED
    )
    test_dataset = (
        pd.read_csv(f"{data_path}/test.csv")
        .sample(frac=TEST_SAMPLE_FRAC, random_state=TEST_SAMPLE_SEED)
        .reset_index(drop=True)
    )

    frames = []

    train_df = train_dataset[["body", "rule", "subreddit", "rule_violation"]].copy()
    train_df = train_df.dropna(subset=["rule", "body"]).reset_index(drop=True)
    frames.append(train_df)

    # ----- Test-time augmentation (instruction-only) -----
    # Positive bodies (violations)
    pos_df = test_dataset[["rule", "subreddit", "positive_example_1", "positive_example_2"]].copy()
    pos1 = pos_df.rename(columns={"positive_example_1": "body"}).drop(columns=["positive_example_2"])
    pos1["rule_violation"] = 1
    pos2 = pos_df.rename(columns={"positive_example_2": "body"}).drop(columns=["positive_example_1"])
    pos2["rule_violation"] = 1

    # Negative bodies (non-violations)
    neg_df = test_dataset[["rule", "subreddit", "negative_example_1", "negative_example_2"]].copy()
    neg1 = neg_df.rename(columns={"negative_example_1": "body"}).drop(columns=["negative_example_2"])
    neg1["rule_violation"] = 0
    neg2 = neg_df.rename(columns={"negative_example_2": "body"}).drop(columns=["negative_example_1"])
    neg2["rule_violation"] = 0

    aug = pd.concat([pos1, pos2, neg1, neg2], axis=0, ignore_index=True)
    aug = aug.dropna(subset=["rule", "body"]).reset_index(drop=True)
    frames.append(aug)

    # ----- Merge, dedup, cap size -----
    dataframe = pd.concat(frames, axis=0, ignore_index=True)
    dataframe = dataframe.drop_duplicates(subset=["rule", "body", "rule_violation"]).reset_index(drop=True)

    if len(dataframe) > MAX_SAMPLES:
        dataframe = dataframe.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)

    return dataframe

def build_dataset(dataframe):
    df = dataframe.copy()
    df["prompt"] = df.apply(build_prompt, axis=1)

    cols = ["prompt"]
    if "rule_violation" in df.columns:
        df["completion"] = df["rule_violation"].map({1: POSITIVE_ANSWER, 0: NEGATIVE_ANSWER})
        cols.append("completion")

    df = df[cols].reset_index(drop=True)
    dataset = Dataset.from_pandas(df)
    dataset.to_pandas().to_csv("/kaggle/working/dataset.csv", index=False)
    return dataset


def main():
    
    dataframe = get_dataframe_to_train(DATA_PATH)
    train_dataset = build_dataset(dataframe)    
    
    lora_config = LoraConfig(
        r=256,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )

    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_use_double_quant = False,
        bnb_4bit_quant_type = 'nf4',
        bnb_4bit_compute_dtype = "float16")

    
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_PATH,
        quantization_config = bnb_config ,
        device_map = {"": f"cuda:{os.environ.get('LOCAL_RANK', 0)}"}
    
    )

    
    training_args = SFTConfig(
        output_dir='outputs_dir',
        num_train_epochs=2,
        
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        
        optim="paged_adamw_8bit",
        learning_rate=1e-4, #keep high, lora usually likes high. 
        weight_decay=0.0,
        max_grad_norm=1.0,
        
        lr_scheduler_type="linear",
        warmup_ratio=0.02,
        
        fp16=True,
        dataloader_pin_memory=True,
        
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},

        logging_steps=200,
        
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        
        seed=42,
        max_length=512,

        ddp_find_unused_parameters=False,

    )
    
    trainer = SFTTrainer(
        model,
        args=training_args,
        train_dataset=train_dataset,
        peft_config=lora_config,
    )
    
    trainer.train()
    trainer.save_model(LORA_PATH)


if __name__ == "__main__":
    main()

Writing train_qwen8b.py


In [9]:
%%writefile accelerate_ds_config_qwen8b.yaml

compute_environment: LOCAL_MACHINE
debug: false
distributed_type: DEEPSPEED
mixed_precision: fp16
use_cpu: false
num_machines: 1
num_processes: 2
machine_rank: 0
same_network: true
main_training_function: main
rdzv_backend: static
downcast_bf16: 'no'
enable_cpu_affinity: false
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false

deepspeed_config:
  fp16:
    enabled: true
    loss_scale: 0
    loss_scale_window: 1000
    initial_scale_power: 16
    hysteresis: 2
    min_loss_scale: 1

  zero_optimization:
    stage: 2
    offload_optimizer:
      device: none
      pin_memory: true
    allgather_partitions: true
    allgather_bucket_size: 2e8
    overlap_comm: true
    reduce_scatter: true
    reduce_bucket_size: 2e8
    contiguous_gradients: true
    round_robin_gradients: true

  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  steps_per_print: 50
  train_batch_size: auto
  train_micro_batch_size_per_gpu: auto
  wall_clock_breakdown: false

Writing accelerate_ds_config_qwen8b.yaml


In [10]:
%%writefile inference_qwen8b.py

import os
import pandas as pd
import vllm
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest

# -------- Config --------
BASE_MODEL_PATH = "/kaggle/input/qwen-3/transformers/8b/1"
LORA_PATH = "output/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = (
    "You are a expert Reddit Moderator, you are given a comment and a rule. "
    "Your task is to classify whether the comment violates the rule. Only respond Yes/No."
)

MAX_PROMPT_TOKENS = 511 

def count_tokens(text: str, tokenizer) -> int:
    return len(tokenizer.encode(text))

def build_prompt(row):
    return f"<|im_start|>system\n{BASE_PROMPT}<|im_end|>\n<|im_start|>user\n<Rule>{row['rule']}</Rule>\n<Comment>{row['body']}</Comment><|im_end|>\n<|im_start|>assistant\n{COMPLETE_PHRASE}"

def build_prompt_with_body_truncation(row, tokenizer, max_tokens: int) -> str:
    """
    Use the exact prompt format, truncating ONLY the body so the prompt fits in `max_tokens`.
    """
    rule = str(row.get("rule", ""))
    body = str(row.get("body", ""))

    head = f"<|im_start|>system\n{BASE_PROMPT}<|im_end|>\n<|im_start|>user\n<Rule>{row['rule']}</Rule>\n<Comment>"
    tail = f"</Comment><|im_end|>\n<|im_start|>assistant\n{COMPLETE_PHRASE}"

    base_tokens = count_tokens(head + tail, tokenizer)
    budget_for_body = max_tokens - base_tokens
    if budget_for_body <= 0:
        body_trunc = ""
    else:
        body_ids = tokenizer.encode(body)
        if len(body_ids) > budget_for_body:
            body_ids = body_ids[:budget_for_body]
        body_trunc = tokenizer.decode(body_ids, skip_special_tokens=True)

    return f"{head}{body_trunc}{tail}"

def main():
    # 1) Load test
    df = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv").reset_index(drop=True)

    # 2) Init vLLM
    llm = vllm.LLM(
        BASE_MODEL_PATH,
        tensor_parallel_size=2,
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=512,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=256,
    )
    tokenizer = llm.get_tokenizer()

    prompts = [
        build_prompt_with_body_truncation(row, tokenizer, MAX_PROMPT_TOKENS)
        for _, row in df.iterrows()
    ]

    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[POSITIVE_ANSWER, NEGATIVE_ANSWER])
    sampling = vllm.SamplingParams(
        skip_special_tokens=True,
        max_tokens=1,
        logits_processors=[mclp],
        logprobs=2,
    )
    outputs = llm.generate(
        prompts,
        sampling_params=sampling,
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH),
    )

    # 5) Take probabilities as-is
    log_probs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    preds = pd.DataFrame(log_probs)[[POSITIVE_ANSWER, NEGATIVE_ANSWER]]
    preds["row_id"] = df["row_id"].values
    preds["rule"] = df["rule"].values
    preds.to_csv("logprobs_qwen3_8b_0_923.csv", index=False)

    # 6) Submission (rank-normalized as before)
    sub = preds[["row_id", "rule", POSITIVE_ANSWER]].rename(columns={POSITIVE_ANSWER: "rule_violation"})
    sub.to_csv("submission_qwen3_8b_0_923.csv", index=False)

if __name__ == "__main__":
    main()

Writing inference_qwen8b.py


# Llama-3.1 8b Instruct DDP
Train and infer with max len 512. Remove duplicates. Use sft DDP.

Training and inference in ~2hrs.

In [11]:
%%writefile train_llama8b.py

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers.utils import is_torch_bf16_gpu_available
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    LogitsProcessor,
    BitsAndBytesConfig
)
import pandas as pd
from datasets import Dataset
import os 


import pandas as pd
from datasets import Dataset
import random, numpy as np
random.seed(42)
np.random.seed(42)

# -----------------------
# Global configuration
# -----------------------
BASE_MODEL_PATH = "/kaggle/input/llama-3.1/transformers/8b-instruct/2"
LORA_PATH = "output/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"
MAX_SAMPLES = 30000
TEST_SAMPLE_FRAC = 1.0
TEST_SAMPLE_SEED = 42

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = (
    "You are a expert Reddit Moderator, you are given a comment and a rule. "
    "Your task is to classify whether the comment violates the rule. Only respond Yes/No."
)

def build_prompt(row):
    return f"""Instructions: {BASE_PROMPT}
<Rule>{row['rule']}</Rule>
<Comment>{row['body']}</Comment>
{COMPLETE_PHRASE}"""

def get_dataframe_to_train(data_path):
    # If Kaggle "public" 10-row test.csv, downsample train a bit for speed.
    if len(pd.read_csv(f"{data_path}/test.csv")) == 10:
        train_frac = 0.05
    else:
        train_frac = 1.0

    train_dataset = pd.read_csv(f"{data_path}/train.csv").sample(
        frac=train_frac, random_state=TEST_SAMPLE_SEED
    )
    test_dataset = (
        pd.read_csv(f"{data_path}/test.csv")
        .sample(frac=TEST_SAMPLE_FRAC, random_state=TEST_SAMPLE_SEED)
        .reset_index(drop=True)
    )

    frames = []

    # ----- Train split (use gold labels) -----
    train_df = train_dataset[["body", "rule", "subreddit", "rule_violation"]].copy()
    train_df = train_df.dropna(subset=["rule", "body"]).reset_index(drop=True)
    frames.append(train_df)

    # ----- Test-time augmentation (instruction-only) -----
    # Positive bodies (violations)
    pos_df = test_dataset[["rule", "subreddit", "positive_example_1", "positive_example_2"]].copy()
    pos1 = pos_df.rename(columns={"positive_example_1": "body"}).drop(columns=["positive_example_2"])
    pos1["rule_violation"] = 1
    pos2 = pos_df.rename(columns={"positive_example_2": "body"}).drop(columns=["positive_example_1"])
    pos2["rule_violation"] = 1

    # Negative bodies (non-violations)
    neg_df = test_dataset[["rule", "subreddit", "negative_example_1", "negative_example_2"]].copy()
    neg1 = neg_df.rename(columns={"negative_example_1": "body"}).drop(columns=["negative_example_2"])
    neg1["rule_violation"] = 0
    neg2 = neg_df.rename(columns={"negative_example_2": "body"}).drop(columns=["negative_example_1"])
    neg2["rule_violation"] = 0

    aug = pd.concat([pos1, pos2, neg1, neg2], axis=0, ignore_index=True)
    aug = aug.dropna(subset=["rule", "body"]).reset_index(drop=True)
    frames.append(aug)

    # ----- Merge, dedup, cap size -----
    dataframe = pd.concat(frames, axis=0, ignore_index=True)
    dataframe = dataframe.drop_duplicates(subset=["rule", "body", "rule_violation"]).reset_index(drop=True)

    if len(dataframe) > MAX_SAMPLES:
        dataframe = dataframe.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)

    return dataframe

def build_dataset(dataframe):
    df = dataframe.copy()
    df["prompt"] = df.apply(build_prompt, axis=1)

    cols = ["prompt"]
    if "rule_violation" in df.columns:
        df["completion"] = df["rule_violation"].map({1: POSITIVE_ANSWER, 0: NEGATIVE_ANSWER})
        cols.append("completion")

    df = df[cols].reset_index(drop=True)
    dataset = Dataset.from_pandas(df)
    dataset.to_pandas().to_csv("/kaggle/working/dataset.csv", index=False)
    return dataset



def main():
  
    dataframe = get_dataframe_to_train(DATA_PATH)
    train_dataset = build_dataset(dataframe)

    lora_config = LoraConfig(
        r=256,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )

    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_use_double_quant = False,
        bnb_4bit_quant_type = 'nf4',
        bnb_4bit_compute_dtype = "float16")

    
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_PATH,
        quantization_config = bnb_config ,
        device_map = {"": f"cuda:{os.environ.get('LOCAL_RANK', 0)}"})

    
    training_args = SFTConfig(
        output_dir='outputs_dir',
        num_train_epochs=2,
        
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        
        optim="paged_adamw_8bit",
        learning_rate=2e-4, #keep high, lora usually likes high. 
        weight_decay=0.0,
        max_grad_norm=1.0,
        
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        
        fp16=True,
        dataloader_pin_memory=True,
        
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},

        logging_steps=200,
        
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        
        seed=42,
        max_length=512,

        ddp_find_unused_parameters=False,

    )
    
    trainer = SFTTrainer(
        model,
        args=training_args,
        train_dataset=train_dataset,
        peft_config=lora_config,
    )
    
    trainer.train()
    trainer.save_model(LORA_PATH)


if __name__ == "__main__":
    main()

Writing train_llama8b.py


In [12]:
%%writefile accelerate_ds_config_llama8b.yaml

compute_environment: LOCAL_MACHINE
debug: false
distributed_type: DEEPSPEED
mixed_precision: fp16
use_cpu: false
num_machines: 1
num_processes: 2
machine_rank: 0
same_network: true
main_training_function: main
rdzv_backend: static
downcast_bf16: 'no'
enable_cpu_affinity: false
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false

deepspeed_config:
  fp16:
    enabled: true
    loss_scale: 0
    loss_scale_window: 1000
    initial_scale_power: 16
    hysteresis: 2
    min_loss_scale: 1

  zero_optimization:
    stage: 2
    offload_optimizer:
      device: none
      pin_memory: true
    allgather_partitions: true
    allgather_bucket_size: 2e8
    overlap_comm: true
    reduce_scatter: true
    reduce_bucket_size: 2e8
    contiguous_gradients: true
    round_robin_gradients: true

  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  steps_per_print: 50
  train_batch_size: auto
  train_micro_batch_size_per_gpu: auto
  wall_clock_breakdown: false

Writing accelerate_ds_config_llama8b.yaml


In [13]:
%%writefile inference_llama8b.py

import os
import pandas as pd
import vllm
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest

# -------- Config --------
BASE_MODEL_PATH = "/kaggle/input/llama-3.1/transformers/8b-instruct/2"
LORA_PATH = "output/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = (
    "You are a expert Reddit Moderator, you are given a comment and a rule. "
    "Your task is to classify whether the comment violates the rule. Only respond Yes/No."
)

MAX_PROMPT_TOKENS = 511 

def count_tokens(text: str, tokenizer) -> int:
    return len(tokenizer.encode(text))

def build_prompt(row):
    # exact template requested
    return (
        f"Instructions: {BASE_PROMPT}\n"
        f"<Rule>{row['rule']}</Rule>\n"
        f"<Comment>{row['body']}</Comment>\n"
        f"{COMPLETE_PHRASE}"
    )

def build_prompt_with_body_truncation(row, tokenizer, max_tokens: int) -> str:
    """
    Use the exact prompt format, truncating ONLY the body so the prompt fits in `max_tokens`.
    """
    rule = str(row.get("rule", ""))
    body = str(row.get("body", ""))

    head = f"Instructions: {BASE_PROMPT}\n<Rule>{rule}</Rule>\n<Comment>"
    tail = f"</Comment>\n{COMPLETE_PHRASE}"

    base_tokens = count_tokens(head + tail, tokenizer)
    budget_for_body = max_tokens - base_tokens
    if budget_for_body <= 0:
        body_trunc = ""
    else:
        body_ids = tokenizer.encode(body)
        if len(body_ids) > budget_for_body:
            body_ids = body_ids[:budget_for_body]
        body_trunc = tokenizer.decode(body_ids, skip_special_tokens=True)

    return f"{head}{body_trunc}{tail}"

def main():
    # 1) Load test
    df = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv").reset_index(drop=True)

    # 2) Init vLLM
    llm = vllm.LLM(
        BASE_MODEL_PATH,
        tensor_parallel_size=2,
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=512,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=256,
    )
    tokenizer = llm.get_tokenizer()

    prompts = [
        build_prompt_with_body_truncation(row, tokenizer, MAX_PROMPT_TOKENS)
        for _, row in df.iterrows()
    ]

    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[POSITIVE_ANSWER, NEGATIVE_ANSWER])
    sampling = vllm.SamplingParams(
        skip_special_tokens=True,
        max_tokens=1,
        logits_processors=[mclp],
        logprobs=2,
    )
    outputs = llm.generate(
        prompts,
        sampling_params=sampling,
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH),
    )

    # 5) Take probabilities as-is
    log_probs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    preds = pd.DataFrame(log_probs)[[POSITIVE_ANSWER, NEGATIVE_ANSWER]]
    preds["rule"] = df["rule"].values
    preds["row_id"] = df["row_id"].values

    # 6) Submission (rank-normalized as before)
    sub = preds[["row_id", "rule",  POSITIVE_ANSWER]].rename(columns={POSITIVE_ANSWER: "rule_violation"})
    sub.to_csv("submission_llama8b_0_926.csv", index=False)

if __name__ == "__main__":
    main()

Writing inference_llama8b.py


# Run Inference 

In [14]:
import time
!python embed_model_lr.py
time.sleep(10)
!python triplet_faiss_public.py
time.sleep(10)
!python deberta_nli_zeroshot.py
time.sleep(10)
!python qwen_32b_inference.py
time.sleep(10)
!python train_deberta.py
time.sleep(10)
!accelerate launch --config_file accelerate_ds_config_qwen8b.yaml train_qwen8b.py
!python inference_qwen8b.py 
!rm -rf output
!accelerate launch --config_file accelerate_ds_config_llama8b.yaml train_llama8b.py
!python inference_llama8b.py #"submission_llama8b_0_926.csv"
!rm -rf output

2025-10-26 17:52:36.541720: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761501156.754173     187 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761501156.814348     187 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading data…

Encoding all for model: e5_base on cuda:1
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00,  2.59it/s]

Encoding all for model: e5_large on cuda:0
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00,  2.38it/s]

Encoding all for model: qwen3_0_6b on cuda:0
Batches: 100%|████████████████████████████████████| 1/1 [00:00<00:00,  2.03it/s]

Encoding all for model: bge_small on cuda:0
Ba

# Ensemble 
Hierarchical Ensemble Blending 

Embedding Model (0.909)
DeBERTa NLI  (0.906)
FAISS Triplet (0.909)
Qwen 32B LLM (0.896)
DeBERTa Public (0.906)
Qwen-3 8b (0.923)
Llama-3.1 8B (0.926)

Blending Pipeline
Per-rule rank normalization: Convert each model's predictions to [0,1] ranks within each rule group to handle different score distributions.
Sequential hierarchical blending:
1. Base: 50% Emb + 50% DeBERTa
2. Add FAISS: 70% prev + 30% FAISS
3. Add Qwen-32B: 65% prev + 35% Qwen-32B
4. Add DeBERTa-Pub: 80% prev + 20% DeBERTa-Pub
5. Add Qwen-1.7B: 80% prev + 20% Qwen-1.7B
6. Add Llama-8B: 75% prev + 25% Llama-8B


Final effective weights (after normalization):

Embedding: ~14.6% 
DeBERTa: ~11.7% 
FAISS: ~8.2% 
Qwen-32B: ~9.6% 
DeBERTa-Pub: ~5.5% 
Qwen-3 8B: ~5.5% 
Llama-3.1 8B: ~44.9% 


Why It Works
✅ Diversity: Blends embedding-based, NLI, triplet, and LLM approaches
✅ Stability: Rank normalization per-rule prevents one model from dominating

Final ensemble achieves ~0.93+ AUC by combining strengths of embedding robustness, LLM reasoning, and triplet learning.

In [15]:
import os
import numpy as np
import pandas as pd

# Paths
path_deberta_private = "/kaggle/working/submission_deberta_nli_0_906.csv"
path_embed_model = "/kaggle/working/submission_embed_model_0_909.csv"
path_faiss_triplet = "/kaggle/working/submission_faiss_triplet_0_909.csv"
qwen_32b = "/kaggle/working/submission_qwen32b_0_896.csv"
deberta_pub = "/kaggle/working/submission_public_0_906.csv"
path_qwen3_8B = "/kaggle/working/submission_qwen3_8b_0_923.csv"
sub_llama8b = "/kaggle/working/submission_llama8b_0_926.csv"

def rank01(s: pd.Series) -> pd.Series:
    """Rank normalize to [0, 1] range"""
    r = s.rank(method="average")
    return (r - r.min()) / (r.max() - r.min() + 1e-12)

def load_csv(path: str, alias: str):
    """Load and validate submission CSV"""
    if not path or not os.path.exists(path):
        print(f"⚠️  {alias}: path missing or not found -> skipping")
        return None
    df = pd.read_csv(path).rename(columns={"rule_violation": alias})
    miss = {"row_id", "rule"} - set(df.columns)
    if miss:
        raise ValueError(f"{alias} missing columns: {miss}")
    print(f"✓ loaded {alias}: {len(df)} rows")
    return df[["row_id", "rule", alias]]

# Load submissions
print("Loading submissions...")
deb = load_csv(path_deberta_private, "deb")
emb = load_csv(path_embed_model, "emb")
faiss = load_csv(path_faiss_triplet, "faiss")
qwen = load_csv(qwen_32b, "qwen32b")
deb_pub = load_csv(deberta_pub, "deb_pub")
qwen3_8b = load_csv(path_qwen3_8B, "qwen3_8b")
llama_8b = load_csv(sub_llama8b, "llama_8b")

if deb is None or emb is None or faiss is None or qwen is None or deb_pub is None or qwen3_8b is None or llama_8b is None:
    raise ValueError("Need all seven submissions: DeBERTa Private, Embedding, FAISS, Qwen-32B, DeBERTa Public, Qwen-1.7B, and llama_8b")

# Align rows (inner join)
print("\nMerging submissions...")
m = emb.merge(deb, on=["row_id", "rule"], how="inner")
m = m.merge(faiss, on=["row_id", "rule"], how="inner")
m = m.merge(qwen, on=["row_id", "rule"], how="inner")
m = m.merge(deb_pub, on=["row_id", "rule"], how="inner")
m = m.merge(qwen3_8b, on=["row_id", "rule"], how="inner")
m = m.merge(llama_8b, on=["row_id", "rule"], how="inner")
print(f"Final dataset: {len(m)} rows")
print(f"Columns: {list(m.columns)}")

# Per-rule rank normalization
print("\nApplying per-rule rank normalization...")
m["r_emb"] = m.groupby("rule")["emb"].transform(rank01).astype(np.float32)
m["r_deb"] = m.groupby("rule")["deb"].transform(rank01).astype(np.float32)
m["r_faiss"] = m.groupby("rule")["faiss"].transform(rank01).astype(np.float32)
m["r_qwen32b"] = m.groupby("rule")["qwen32b"].transform(rank01).astype(np.float32)
m["r_deb_pub"] = m.groupby("rule")["deb_pub"].transform(rank01).astype(np.float32)
m["r_qwen3_8b"] = m.groupby("rule")["qwen3_8b"].transform(rank01).astype(np.float32)
m["r_llama_8b"] = m.groupby("rule")["llama_8b"].transform(rank01).astype(np.float32)

# Hierarchical blending
print("\nBuilding ensemble...")

# Base: 50-50 emb–deb
final = 0.5 * m["r_emb"] + 0.5 * m["r_deb"]
weights = {"emb": 0.5, "deb": 0.5}

# Add FAISS Triplet: 30%
final = 0.7 * final + 0.3 * m["r_faiss"]
for k in list(weights.keys()):
    weights[k] *= 0.7
weights["faiss"] = 0.3

# Add Qwen-32B: 35%
final = 0.65 * final + 0.35 * m["r_qwen32b"]
for k in list(weights.keys()):
    weights[k] *= 0.65
weights["qwen32b"] = 0.35

# Add DeBERTa Public: 20%
final = 0.8 * final + 0.2 * m["r_deb_pub"]
for k in list(weights.keys()):
    weights[k] *= 0.8
weights["deb_pub"] = 0.2

# Add Qwen-1.7B: 20%
final = 0.8 * final + 0.2 * m["r_qwen3_8b"]
for k in list(weights.keys()):
    weights[k] *= 0.8
weights["qwen3_8b"] = 0.2

# Add Qwen-4B on top: 10%
final = 0.75 * final + 0.25 * m["r_llama_8b"]
for k in list(weights.keys()):
    weights[k] *= 0.75
weights["llama_8b"] = 0.25

# Print effective weights
s = sum(weights.values())
weights_normalized = {k: round(v/s, 6) for k, v in weights.items()}
print(f"Effective weights (rank-normalized): {weights_normalized}")

# Clip to valid probability range
final = np.clip(final.to_numpy(np.float32), 1e-6, 1-1e-6)

# Save submission
print("\nSaving submission...")
sub = m[["row_id"]].copy()
sub["rule_violation"] = final
sub.to_csv("submission.csv", index=False)
print(f"✅ Saved submission.csv | {len(sub)} rows")
print(f"Prediction stats: mean={final.mean():.4f}, min={final.min():.4f}, max={final.max():.4f}")
print(f"\nFinal weights: Emb={weights_normalized['emb']}, Deb={weights_normalized['deb']}, FAISS={weights_normalized['faiss']}, Qwen32B={weights_normalized['qwen32b']}, DeBPub={weights_normalized['deb_pub']}, Qwen 3 8b={weights_normalized['qwen3_8b']}, llama_8b={weights_normalized['llama_8b']}")

Loading submissions...
✓ loaded deb: 10 rows
✓ loaded emb: 10 rows
✓ loaded faiss: 10 rows
✓ loaded qwen32b: 10 rows
✓ loaded deb_pub: 10 rows
✓ loaded qwen3_8b: 10 rows
✓ loaded llama_8b: 10 rows

Merging submissions...
Final dataset: 10 rows
Columns: ['row_id', 'rule', 'emb', 'deb', 'faiss', 'qwen32b', 'deb_pub', 'qwen3_8b', 'llama_8b']

Applying per-rule rank normalization...

Building ensemble...
Effective weights (rank-normalized): {'emb': 0.1092, 'deb': 0.1092, 'faiss': 0.0936, 'qwen32b': 0.168, 'deb_pub': 0.12, 'qwen3_8b': 0.15, 'llama_8b': 0.25}

Saving submission...
✅ Saved submission.csv | 10 rows
Prediction stats: mean=0.4500, min=0.0000, max=0.8649

Final weights: Emb=0.1092, Deb=0.1092, FAISS=0.0936, Qwen32B=0.168, DeBPub=0.12, Qwen 3 8b=0.15, llama_8b=0.25
